# Reproducing Kim, Lycke et al. (2026) from dandiset 001868

Paper. Kim R, Lycke R, Zolotavin P, Montes J, Xie C, Luan L (2026). Learning induces
activation-mechanism-dependent neural plasticity in an intracortical microstimulation task.
*Science Advances* 12, eaef0343. [doi:10.1126/sciadv.aef0343](https://doi.org/10.1126/sciadv.aef0343).
The claims file (`claims/001868/10.1126-sciadv.aef0343.yaml`) was written from the version of
record (PMC13544205) with its supplement; this attempt read the same text.

Dandiset. [DANDI:001868](https://dandiarchive.org/dandiset/001868/0.260715.2016), version
0.260715.2016, pinned because the paper's data availability statement cites it
(doi:10.48324/dandi.001868/0.260715.2016), as the claims file does. It is the only published
version; the draft has the same 85 assets and size and was last modified at publication
(2026-07-15), so there is no other version to compare.

What the dandiset holds. 85 NWB files of 12 mice. Six behavioural mice (ICMS92, 93, 98, 100,
101 and 83, the paper's Mice 1 to 6) have 55 sessions with sorted units (spike times, one mean
waveform, cell type, position on the probe), trial and train tables, wheel position, and in most
sessions volumetric and planar per-ROI dF/F with ROI masks. Six control mice have 30 files with
survey imaging (ROI masks and binary activation per channel and current) and, in some, sorted
units. The behavioural files hold no stimulation-site position in the imaging volume, no
activation calls, and no identity of planar ROIs across sessions; no file holds spike waveforms
in and out of stimulation or multi-channel templates.

Method details from the paper's code. Where the claims file leaves a detail open, it is taken
from the code the paper links (XieLuanLab/icms-activation-plasticity at d1922d1), each cited where it
is used: the threshold at 50% of the fitted function (processing/dataloader.py), the pulse-locking
null drawn from 20% of the pulses (responses_v2/pulse_locked_response_metrics.py), the position of
the stimulation contact, (32 - stim_channel) x 60 um, and the cortical depths of fig. S4
(longitudinal_data_utils.py), and the control pipeline (processing/control, python/fig_s7,
processing/pop_coupling/pop_coupling_control.py). The code is a source of method only; no data come
from it.

Random numbers. Every random draw uses its own `numpy` generator with a fixed seed, given in
the cell that draws.

Outside inputs. None.

| Claim | Status |
|---|---|
| fig2a-threshold-decrease | tested |
| fig2a-steepest-decline-early | tested |
| figs1e-response-times-decrease | tested |
| fig2c-count-increase-fixed-current, fig2b-example-counts, figs2ab-density-peak-200um, fig2e-density-200um-increase, fig2e-radius-increase, figs2de-behaviour-increase-comparison, fig2g-stable-at-threshold | blocked: no stimulation-site position in the imaging volume and no activation calls |
| figs2de-control-stable | blocked: no voxel size, plane spacing or channel-to-contact mapping for the control survey, so no density; the count half is run and reported |
| fig3b, fig3df, fig3i, fig3j (two), fig3lm, fig3n (two), figs6 (planar tracking) | blocked: no identity of planar ROIs across sessions |
| fig4-unit-counts | tested |
| fig4d-waveform-separation | blocked: no spike waveforms in and out of stimulation |
| figs3b, figs3c, figs3d, fig4g, fig4ij (two) | tested |
| fig5d, fig5e, figs4b, fig5fg, figs4de, fig5h | tested |
| figs5c-wheel-onset-decrease, figs5-restricted-window-consistent | tested |
| figs7-control-no-change | tested, with the paper's control code |
| figs8-tracked-units | blocked: no multi-channel templates or tracked-unit labels |
| fig6c, fig6d, fig6f, figs9a | tested |
| figs9-coupling-window-and-control | tested: the 120 ms part, and the control part with the paper's control code |

In [ ]:
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"  # many small fits; one linear-algebra thread each (protocol, Keep it fast)
import pickle
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import stats

import icms

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
if os.environ.get("ICMS_DEV_PICKLE"):  # development only: a cached copy of the same loaded data
    sessions_all = pickle.load(open(os.environ["ICMS_DEV_PICKLE"], "rb"))
else:
    sessions_all = icms.load_behaviour_sessions(threads=12)
print(f"loaded {len(sessions_all)} behavioural sessions from dandiset {icms.DANDISET} "
      f"version {icms.VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
assets = icms.list_assets()
kinds = Counter()
for a in assets:
    subj = a["path"].split("/")[0][4:]
    kinds[(subj in icms.BEHAVIOUR, a["path"].split("_")[-1][:-4])] += 1
print("files:", len(assets), "| behavioural mice:", sum(v for (b, _), v in kinds.items() if b),
      "| control mice:", sum(v for (b, _), v in kinds.items() if not b))
for (b, k), v in sorted(kinds.items()):
    print(f"  {'behaviour' if b else 'control'} files of kind {k}: {v}")

tab = pd.DataFrame([{"subject": s["subject"], "mouse": icms.MOUSE[s["subject"]], "sex": s["sex"],
                     "date": s["date"], "day": s["day"], "week": s["week"],
                     "units": len(s["spikes"]), "trials": len(s["trials"]["trial_index"]),
                     "trains_ephys": len(icms.ephys_trains(s)["start_time"]),
                     "catch": int((s["trials"]["current_uA"] == 0).sum()),
                     "catch_hit": int(s["trials"]["is_hit"][s["trials"]["current_uA"] == 0].sum()),
                     "catch_rt": int((~np.isnan(s["trials"]["response_time"][s["trials"]["current_uA"] == 0])).sum()),
                     "currents": ",".join(str(int(c)) for c in np.unique(s["trials"]["current_uA"])),
                     "channels": ",".join(str(c) for c in np.unique(s["trials"]["stim_channel"]) if c),
                     "wheel": "wheel" in s} for s in sessions_all])
print(tab.to_string())
print("\nsessions per mouse and week:")
print(pd.crosstab(tab["mouse"], tab["week"]))
sessions = [s for s in sessions_all if s["week"] <= 4]
print(f"\nsessions in weeks 0 to 4: {len(sessions)} (the paper: 46 sessions); weeks above 4 dropped: "
      f"{len(sessions_all) - len(sessions)}")
print("catch trials:", tab["catch"].sum(), "| catch trials marked hit:", tab["catch_hit"].sum(),
      "| catch trials with a response time:", tab["catch_rt"].sum())
print("units by stored cell_type:", dict(Counter(ct for s in sessions for ct in s["cell_type"])))

In [ ]:
# Trains and trials: units, ranges and the shared clock
rows = []
for s in sessions_all:
    st, T = s["stim"], s["trials"]
    ti = dict(zip(T["trial_index"], range(len(T["trial_index"]))))
    lag = np.array([st["start_time"][i] - T["start_time"][ti[k]] for i, k in enumerate(st["trial_index"])])
    tr = icms.ephys_trains(s)
    rows.append({"session": f'{s["subject"]} {s["date"]}', "trains": len(st["start_time"]),
                 "trains_70_in_ephys": len(tr["start_time"]),
                 "pulse_counts": ",".join(map(str, sorted(set(st["pulse_count"].tolist())))) if len(st["pulse_count"]) < 0 else
                 f'{st["pulse_count"].min()}-{st["pulse_count"].max()}',
                 "freq": ",".join(map(str, sorted(set(st["frequency_hz"].tolist())))),
                 "phase_us": ",".join(map(str, sorted(set(st["pulse_width_us"].tolist())))),
                 "stim_minus_trial_ms": f"{1000 * lag.min():.1f} to {1000 * lag.max():.1f}",
                 "rt_hit_s": f'{np.nanmin(T["response_time"]):.3f} to {np.nanmax(T["response_time"]):.3f}',
                 "ephys_end_s": round(s["ephys_end"], 1), "last_train_s": round(float(st["stop_time"].max()), 1)})
chk = pd.DataFrame(rows)
print(chk.to_string())
long_out = sum(((s["stim"]["pulse_count"] != 70) & (s["stim"]["stop_time"] < s["ephys_end"])).sum() for s in sessions_all)
print("trains that are not 70 pulses but lie inside the electrophysiology recording:", long_out)
print("hit trials with no response time:",
      sum(int(np.isnan(s["trials"]["response_time"][s["trials"]["is_hit"]]).sum()) for s in sessions_all),
      "| miss trials with a response time:",
      sum(int((~np.isnan(s["trials"]["response_time"][~s["trials"]["is_hit"]])).sum()) for s in sessions_all))
dur = np.concatenate([1000 * (icms.ephys_trains(s)["stop_time"] - icms.ephys_trains(s)["start_time"]) for s in sessions_all])
print(f"70-pulse trains: stop - start from {dur.min():.2f} to {dur.max():.2f} ms (median {np.median(dur):.2f}); "
      "70 pulses 10 ms apart span 690 ms plus the last pulse; no per-pulse times are stored")

In [ ]:
# Shared clock: spikes and wheel against the train onsets, with the onsets shifted as a contrast.
rng = np.random.default_rng(100)
clock = []
for s in sessions:
    tr = icms.ephys_trains(s)
    on = np.sort(tr["start_time"][np.isin(tr["current_uA"], icms.CURRENTS)])
    if len(on) < 10:
        continue
    sh = on + rng.uniform(1.5, 2.5, len(on))  # shifted into the intertrial period
    allspk = np.sort(np.concatenate(s["spikes"]))
    d_true = (icms.counts_in(allspk, on, 0, 0.7) - icms.counts_in(allspk, on, -0.7, 0)).mean()
    d_shift = (icms.counts_in(allspk, sh, 0, 0.7) - icms.counts_in(allspk, sh, -0.7, 0)).mean()
    T = s["trials"]
    hit = T["is_hit"] & (T["current_uA"] > 0)
    miss = ~T["is_hit"] & (T["current_uA"] > 0)
    w_hit = np.nanmedian(np.abs(icms.wheel_displacement(s, T["start_time"][hit], 0, 0.7)[:, -1])) if "wheel" in s else np.nan
    w_miss = np.nanmedian(np.abs(icms.wheel_displacement(s, T["start_time"][miss], 0, 0.7)[:, -1])) if "wheel" in s else np.nan
    w_pre_hit = np.nanmedian(np.abs(icms.wheel_displacement(s, T["start_time"][hit] - 0.7, 0, 0.7)[:, -1])) if "wheel" in s else np.nan
    clock.append({"session": f'{s["subject"]} {s["date"]}', "delta_spikes_true": d_true,
                  "delta_spikes_shifted": d_shift, "wheel_hit_0_700ms": w_hit, "wheel_miss_0_700ms": w_miss,
                  "wheel_hit_before_onset": w_pre_hit})
clock = pd.DataFrame(clock)
print(clock.round(2).to_string())
print("sessions where the train raises the summed spike count more than the shifted onsets:",
      int((clock.delta_spikes_true > clock.delta_spikes_shifted).sum()), "of", len(clock))
print("sessions where hit trials move the wheel more than miss trials in the 700 ms after trial start:",
      int((clock.wheel_hit_0_700ms > clock.wheel_miss_0_700ms).sum()), "of", len(clock))

In [ ]:
# Recordings deposited twice: identical trial tables or spike trains across files
sig_t, sig_u = defaultdict(list), defaultdict(list)
for s in sessions_all:
    sig_t[hash(np.round(s["trials"]["start_time"], 4).tobytes())].append(f'{s["subject"]} {s["date"]}')
    for i, spk in enumerate(s["spikes"]):
        if len(spk) > 20:
            sig_u[hash(np.round(spk[:200], 5).tobytes())].append(f'{s["subject"]} {s["date"]} unit {i}')
print("identical trial tables:", [v for v in sig_t.values() if len(v) > 1])
dup_u = [v for v in sig_u.values() if len(v) > 1]
print("identical spike trains (first 200 spikes):", len(dup_u), dup_u[:10])

In [ ]:
# Cell type: the stored label against the trough-to-peak of the stored mean waveform
# (below 0.45 ms is a narrow-waveform interneuron; Fig. 4H legend, claims common method)
def trough_to_peak_ms(w, fs=30000.0):
    i = int(np.argmin(w))
    j = i + int(np.argmax(w[i:]))
    return (j - i) / fs * 1000.0

ct_rows = []
for s in sessions:
    for i, w in enumerate(s["waveform"]):
        ct_rows.append({"mouse": icms.MOUSE[s["subject"]], "stored": s["cell_type"][i],
                        "t2p_ms": trough_to_peak_ms(w)})
ct = pd.DataFrame(ct_rows)
ct["t2p_class"] = np.where(ct.t2p_ms < 0.45, "interneuron", "pyramidal")
print(pd.crosstab(ct.stored, ct.t2p_class))
print("trough-to-peak range (ms):", ct.t2p_ms.min().round(3), "to", ct.t2p_ms.max().round(3))
print("waveform_mean shape per unit:", sessions[0]["waveform"].shape[1], "samples (3 ms at 30 kHz), one channel")
print("unit_y_um range:", np.nanmin(np.concatenate([s["unit_y"] for s in sessions])).round(1), "to",
      np.nanmax(np.concatenate([s["unit_y"] for s in sessions])).round(1), "um; electrode rel_y range:",
      min(s["el_y"].min() for s in sessions), "to", max(s["el_y"].max() for s in sessions), "um")
_f = icms.open_asset(assets[[a["path"] for a in assets].index(sessions[0]["path"])]["asset_id"])
for _k in ["units/cell_type", "units/waveform_mean", "units/accepted", "units/unit_y_um",
           "general/extracellular_ephys/electrodes/channel_name", "intervals/electrical_stimulation/stim_channel"]:
    print(f"description of {_k}: {_f[_k].attrs.get('description', '')!r}")
_f.close()
acc = pd.DataFrame([{"session": f'{s["subject"]} {s["date"]}', "units": len(s["accepted"]),
                     "accepted": int(s["accepted"].sum())} for s in sessions])
print("units not accepted (weeks 0 to 4):", int((acc.units - acc.accepted).sum()), "of", int(acc.units.sum()),
      "| sessions with any:", acc[acc.units > acc.accepted].to_dict("records"))

In [ ]:
# Imaging tables of the behavioural files: ROI counts, segmentation descriptions, reference images,
# mask layout (metadata and shapes only). The blocked imaging claims rest on these outputs.
from concurrent.futures import ThreadPoolExecutor

img_assets = [a for a in assets if "behavior" in a["path"] and "ophys" in a["path"]]
with ThreadPoolExecutor(12) as _ex:
    IMG = pd.DataFrame(list(_ex.map(icms.imaging_metadata, img_assets))).sort_values(["subject", "date"])
IMG["mouse"] = IMG.subject.map(icms.MOUSE)
print(IMG[["mouse", "date", "Planar_rois", "Volumetric_rois", "Volumetric_mask_planes"]].to_string(index=False))
print("files with imaging:", len(IMG), "| with planar ROIs:", int((IMG.Planar_rois > 0).sum()))
print("planar ROIs per session, Mouse 1:", IMG.loc[(IMG.mouse == "Mouse 1") & (IMG.Planar_rois > 0), "Planar_rois"].tolist())
print("planar segmentation description (one file):", repr(IMG.Planar_description.dropna().iloc[0]))
print("volumetric segmentation description (one file):", repr(IMG.Volumetric_description.dropna().iloc[0]))
print("planar columns:", IMG.Planar_columns.dropna().unique().tolist(), "| volumetric columns:",
      IMG.Volumetric_columns.dropna().unique().tolist())
_pl = IMG[IMG.Planar_rois > 0]
print("planar roi_id_source is the row index 0..N-1 in every session:", bool(_pl.Planar_roi_id_is_row_index.astype(bool).all()))
print("reference images stored in the segmentations: planar", sorted(_pl.Planar_reference_images.astype(int).unique().tolist()),
      "| volumetric", sorted(IMG.Volumetric_reference_images.dropna().astype(int).unique().tolist()))
print("processing/ophys entries:", sorted({e for v in IMG.ophys_entries for e in v}))
print("imaging planes:", sorted({e for v in IMG.imaging_planes for e in v}))


## fig2a-threshold-decrease

> Behavioural detection thresholds fall with learning; thresholds in the late phase (weeks 2 to 4)
> are lower than in the early phase (weeks 0 to 1), median 5.5 against 4.2 uA.

Per channel and block, a cumulative Gaussian with free guess and lapse is fitted with psignifit 4
at its default grid to the hit proportion at each current, with the block's catch-trial false-alarm
rate as a 0 uA point; the session threshold of a channel is the mean of its block thresholds. The
threshold is the current at which the fitted function reaches 50% detection (`ps_x50`), as the
criterion says and as the paper's code takes it (`ps.getThreshold(result, 0.5)` in
processing/dataloader.py, whose default is the scaled function). psignifit's own threshold
parameter (the midpoint of the unscaled sigmoid) and a bounded maximum likelihood fit (its mu and
its x50) are shown as variants. The dataset stores no thresholds, so only refitted ones are shown.
Mouse 6 (ICMS83) has no catch trials in the files; its fits have no 0 uA point. Effect size
r = |Z| / sqrt(n1 + n2) (claims, common method, for thresholds).

In [ ]:
def r_from_p(p, n1, n2):
    return abs(stats.norm.ppf(p / 2)) / np.sqrt(n1 + n2)

from concurrent.futures import ProcessPoolExecutor
t0 = time.time()
items = []
for s in sessions:
    for d in icms.block_data(s):
        items.append(({"subject": s["subject"], "mouse": icms.MOUSE[s["subject"]], "date": s["date"],
                       "week": s["week"], "phase": s["phase"]}, d))
with ProcessPoolExecutor(4) as ex:
    fits = list(ex.map(icms.fit_block, [d for _, d in items], chunksize=4))
thr_rows = [{**meta, **f} for (meta, _), f in zip(items, fits)]
print(f"{len(thr_rows)} block fits in {time.time() - t0:.0f} s")
blk = pd.DataFrame(thr_rows)
print("block fits:", len(blk), "| with a catch point:", int(blk.has_catch.sum()),
      "| psignifit x50 range", blk.ps_x50.min().round(2), "to", blk.ps_x50.max().round(2),
      "| x50 undefined (guess above 0.5):", int(blk.ps_x50.isna().sum()),
      "| MLE mu at a bound (0 or 30 uA):", int(((blk.mu < 0.01) | (blk.mu > 29.99)).sum()),
      "| guess range", blk.guess.min().round(3), "to", blk.guess.max().round(3),
      "| lapse range", blk.lapse.min().round(3), "to", blk.lapse.max().round(3))
print("blocks per channel and session:", blk.groupby(["subject", "date", "channel"]).size().value_counts().to_dict())
thr = blk.groupby(["subject", "mouse", "date", "week", "phase", "channel"], as_index=False).agg(
    ps=("ps_x50", "mean"), ps_u=("ps_threshold", "mean"), mu=("mu", "mean"), x50=("x50", "mean"),
    n_blocks=("block", "size"))
THR_NAMES = {"ps": "psignifit x50 (main)", "ps_u": "psignifit unscaled threshold", "mu": "MLE mu", "x50": "MLE x50"}
print("channel-session thresholds:", len(thr), "| range (psignifit):", thr.ps.min().round(2), "to", thr.ps.max().round(2), "uA")
res_thr = {}
for name in THR_NAMES:
    e, l = thr.loc[thr.phase == "early", name], thr.loc[thr.phase == "late", name]
    t = icms.mwu(e, l)
    res_thr[name] = t
    print(f"{THR_NAMES[name]}: early median {t['med1']:.2f} uA (n={t['n1']}), late median {t['med2']:.2f} uA (n={t['n2']}), "
          f"two-sided MWU p = {t['p']:.3g}, r = {r_from_p(t['p'], t['n1'], t['n2']):.2f}")
print("\nby mouse (psignifit x50):")
print(thr.groupby(["mouse", "phase"]).ps.agg(["median", "size"]).unstack().round(2))
# without Mouse 6, whose blocks lack catch trials
t6 = icms.mwu(thr.loc[(thr.phase == "early") & (thr.subject != "ICMS83"), "ps"],
              thr.loc[(thr.phase == "late") & (thr.subject != "ICMS83"), "ps"])
print(f"\nwithout Mouse 6: early {t6['med1']:.2f} (n={t6['n1']}), late {t6['med2']:.2f} (n={t6['n2']}), p = {t6['p']:.3g}")

**Result: reproduced.** With psignifit at its default grid, the median channel-session threshold (the current at 50% of the fitted function) is 5.39 uA early (n = 46) and 3.88 uA late (n = 89), both within tolerance (late 3.78 to 4.62), and late is lower (two-sided Mann-Whitney p = 5.32e-06, r = 0.39). The early n is within 43 +- 4; the late n of 89 is one above 80 +- 8, which the criterion does not judge. psignifit's unscaled threshold gives 5.41 and 3.84 uA. The bounded maximum likelihood variant, which is not the criterion's fit, puts the late median below the tolerance (mu 3.71, x50 3.70 uA) with the same direction (p = 4.9e-07, 5.15e-06). Every mouse is lower late. Without Mouse 6, whose blocks have no catch trials, early 5.17 and late 3.91 uA (p = 7.61e-05).

## fig2a-steepest-decline-early

> The steepest fall in threshold is between weeks 0 and 1, followed by smaller changes and
> relative stability in weeks 2 to 4.

In [ ]:
wk = thr.groupby("week").ps.agg(["median", "size"])
wk["change_from_previous"] = wk["median"].diff()
print(wk.round(3))
for name in THR_NAMES:
    m = thr.groupby("week")[name].median()
    d = m.diff()
    print(f"{THR_NAMES[name]}: week medians {m.round(3).to_dict()} | fall week 0 to 1 {-d.loc[1]:.3f} uA, "
          f"largest later change {d.loc[2:].abs().max():.3f} uA (weeks {int(d.loc[2:].abs().idxmax()) - 1} to "
          f"{int(d.loc[2:].abs().idxmax())})")
print("by mouse, median psignifit x50 per week:")
print(thr.pivot_table(index="mouse", columns="week", values="ps", aggfunc="median").round(2))

**Result: reproduced, on the criterion's reading of the figure.** With the psignifit thresholds of fig2a, the median falls by 1.837 uA from week 0 to week 1; the largest later week-to-week change is 0.792 uA (week 1 to 2), and weeks 2 to 4 stay between 3.808 and 4.310 uA. psignifit's unscaled threshold gives 1.775 against 0.846 uA. The maximum likelihood variant reverses the order (mu 1.016 against 1.186 uA, x50 1.139 against 1.257 uA, both for week 1 to 2); it is not the criterion's fit, but the ordering depends on the fitting method.

## figs1e-response-times-decrease

> Response times follow the same time course as thresholds, falling over training.

Response time of hit trials at 4 to 6 uA. The dataset's `response_time` is measured from trial
start, which precedes the train onset by 0 to 85 ms (data check); the time from train onset is
shown as a variant.

In [ ]:
rt_rows = []
for s in sessions:
    T = s["trials"]
    st = s["stim"]
    on = dict(zip(st["trial_index"], st["start_time"]))
    m = T["is_hit"] & np.isin(T["current_uA"], icms.CURRENTS) & ~np.isnan(T["response_time"])
    for k, t0, rt in zip(T["trial_index"][m], T["start_time"][m], T["response_time"][m]):
        rt_rows.append({"subject": s["subject"], "mouse": icms.MOUSE[s["subject"]], "week": s["week"],
                        "phase": s["phase"], "rt_ms": 1000 * rt,
                        "rt_from_train_ms": 1000 * (t0 + rt - on.get(k, np.nan))})
rt = pd.DataFrame(rt_rows)
print("hit trials at 4 to 6 uA:", len(rt), "| range", rt.rt_ms.min().round(1), "to", rt.rt_ms.max().round(1), "ms")
for col in ["rt_ms", "rt_from_train_ms"]:
    per = rt.pivot_table(index="mouse", columns="week", values=col, aggfunc="median")
    late = rt[rt.phase == "late"].groupby("mouse")[col].median()
    med = per.median()
    print(f"\n{col}: median per mouse and week")
    print(per.round(1))
    print("median over mice per week:", med.round(1).to_dict(), "| late phase:", round(late.median(), 1))
    fall = med[0] - late.median()
    print(f"fall week 0 to late: {fall:.1f} ms; fall week 0 to 1: {med[0] - med[1]:.1f} ms "
          f"({100 * (med[0] - med[1]) / fall:.0f}% of the fall)" if fall > 0 else f"no fall: {fall:.1f} ms")
    t = icms.mwu(rt.loc[rt.phase == "early", col], rt.loc[rt.phase == "late", col])
    print(f"variant, MWU early vs late hit trials: early {t['med1']:.1f} (n={t['n1']}), late {t['med2']:.1f} "
          f"(n={t['n2']}), p = {t['p']:.3g}, rank-biserial r = {t['r']:.2f}")

**Result: reproduced.** The median over mice of the per-mouse median hit response time is 274.9 ms in week 0 and 193.9 ms over the late phase; 75.7 of the 81.0 ms fall (93%) is between weeks 0 and 1. Measured from the train onset the values are 265.4 and 186.8 ms (93% again). Over hit trials, early 190.1 ms against late 173.8 ms (p = 7.19e-17).

## figs5c-wheel-onset-decrease

> Wheel-movement onset decreases from early to late training; median onset is 122 ms in the
> fastest animal (Mouse 1), 135 ms over all hit trials and 129 ms in the late phase.

The stored `wheel_position_processed` is already low-passed at 20 Hz forward and backward (its
description). The response threshold is the 25th percentile of absolute displacement at response
time; the dataset's miss trials have no response time (98 of them do, data check), so the hit
trials define it, and the miss trials with a response time are added as a variant. Onset is
measured from the train onset, the first time |displacement| exceeds 50% of the threshold for 50 ms.
Mice 1 to 5, hit trials at 4 to 6 uA. Readings: threshold per animal and pooled.

In [ ]:
wrows = []
for s in sessions:
    if s["subject"] == "ICMS83":
        continue
    T = s["trials"]
    on = dict(zip(s["stim"]["trial_index"], s["stim"]["start_time"]))
    m = np.isin(T["current_uA"], icms.CURRENTS) & ~np.isnan(T["response_time"])
    for k, t0, rt_, h in zip(T["trial_index"][m], T["start_time"][m], T["response_time"][m], T["is_hit"][m]):
        if k not in on:
            continue
        wrows.append({"si": id(s), "s": s, "subject": s["subject"], "mouse": icms.MOUSE[s["subject"]],
                      "week": s["week"], "phase": s["phase"], "hit": bool(h), "onset_t": on[k],
                      "resp_after_onset": t0 + rt_ - on[k]})
W = pd.DataFrame(wrows)
disp = np.vstack([icms.wheel_displacement(r.s, [r.onset_t], 0, 1.0)[0] for r in W.itertuples()])
ridx = np.clip(np.round(W.resp_after_onset.values * 1000).astype(int), 0, disp.shape[1] - 1)
W["abs_at_resp"] = np.abs(disp[np.arange(len(W)), ridx])
print("trials whose wheel trace is missing (outside the stored signal):", int(np.isnan(disp).all(1).sum()))
print("trials:", len(W), "| hit:", int(W.hit.sum()), "| |displacement| at response, range:",
      W.abs_at_resp.min().round(1), "to", W.abs_at_resp.max().round(1), "a.u.")
wheel_res = {}
for reading in ["per animal", "pooled"]:
    for which in ["hit only", "hit and miss with a response time"]:
        base = W[W.hit] if which == "hit only" else W
        if reading == "pooled":
            thr_w = {sub: np.nanpercentile(base.abs_at_resp, 25) for sub in W.subject.unique()}
        else:
            thr_w = base.groupby("subject").abs_at_resp.quantile(0.25).to_dict()
        hits = W[W.hit].copy()
        hd = disp[W.hit.values]
        hits["onset_ms"] = [1000 * icms.movement_onset(hd[i:i + 1], thr_w[sub])[0]
                            for i, sub in enumerate(hits.subject)]
        m1 = hits.loc[hits.subject == "ICMS92", "onset_ms"].median()
        allm = hits.onset_ms.median()
        latem = hits.loc[hits.phase == "late", "onset_ms"].median()
        t = icms.mwu(hits.loc[hits.phase == "early", "onset_ms"], hits.loc[hits.phase == "late", "onset_ms"])
        wheel_res[(reading, which)] = (m1, allm, latem, t)
        print(f"\nthreshold {reading}, from {which}: thresholds {dict((k, round(v, 1)) for k, v in thr_w.items())}")
        print(f"  median onset Mouse 1 {m1:.1f} ms, all hit trials {allm:.1f} ms, late {latem:.1f} ms; "
              f"no onset found in {int(hits.onset_ms.isna().sum())} of {len(hits)}")
        print(f"  early {t['med1']:.1f} ms (n={t['n1']}) vs late {t['med2']:.1f} ms (n={t['n2']}), "
              f"p = {t['p']:.3g}, rank-biserial r = {t['r']:.2f}")
        no_on = hits.onset_ms.isna().values
        print(f"  of the hit trials with no onset, {int((no_on & np.isnan(hd).all(1)).sum())} have no stored wheel trace "
              f"and {int((no_on & ~np.isnan(hd).all(1)).sum())} have one")
        print("  median per mouse:", hits.groupby("mouse").onset_ms.median().round(1).to_dict())

**Result: reproduced.** With the response threshold per animal, the median onset is 119.0 ms in Mouse 1, 133.0 ms over all hit trials and 128.0 ms in the late phase, all within tolerance, and onset is earlier late than early (144.0 against 128.0 ms, n = 1307 and 2897 against the paper's 1305 and 2922; p = 6.09e-32, r = -0.23). The pooled threshold gives 118.0, 134.0 and 129.0 ms (p = 1.94e-29). The 568 of 4772 hit trials with no onset are the 568 trials with no stored wheel trace; every hit trial with a trace has an onset.

## Electrophysiology: responses

One response is one unit at one stimulation channel and current in one session (claims, common
method). All currents are computed, since fig. S3D needs the threshold current; the other claims
use 4, 5 and 6 uA. `mod50` and `mod100` are the modulated calls with the code's filters and a
z cap of 50 or 100; `mod_methods` uses the Methods' filters alone. Pulse-locking is computed for
every response modulated under `mod100` (a superset of `mod50`), as the paper's code computes it
(`icms.pulse_locking`, from processing/batch_process/postprocessing/responses_v2/
pulse_locked_response_metrics.py at d1922d1): 500 draws of 20% of the pulses for the spike
probability curve, and a null of 100 uniform placements of the spikes of one 20% draw. Pulses are
taken 10 ms apart from the train onset, as the files store train onsets only.

In [ ]:
t0 = time.time()
ch_days = icms.channel_days(sessions)
resp_rows = icms.response_table(sessions, rng_seed=0, with_trials=True)
for r in resp_rows:
    r["mod50"] = icms.modulated(r, 50, True, ch_days)
    r["mod100"] = icms.modulated(r, 100, True, ch_days)
    r["mod_methods"] = icms.modulated(r, code_filters=False)
    r["mod"] = r["mod100"]
    r["ctype"] = {"pyramidal": "pyramidal", "Pyramidal Cell": "pyramidal", "interneuron": "interneuron",
                  "Narrow Interneuron": "interneuron"}.get(str(r["cell_type"]), "unknown")
icms.add_pulse_locking(resp_rows, sessions, rng_seed=1)
R = pd.DataFrame([{k: v for k, v in r.items() if k not in ("trial_delta", "trial_hit", "starts")} for r in resp_rows])
R["mouse"] = R.subject.map(icms.MOUSE)
# the criterion's rule applied to the stored mean waveform (variant; see the data check)
R["t2p_type"] = ["interneuron" if trough_to_peak_ms(sessions[si]["waveform"][u]) < 0.45 else "pyramidal"
                 for si, u in zip(R.si, R.unit)]
print(f"responses: {len(R)} in {time.time() - t0:.0f} s; modulated mod50 {int(R.mod50.sum())}, mod100 "
      f"{int(R.mod100.sum())}, Methods filters only {int(R.mod_methods.sum())}")
print("channels dropped by the 5-day filter:", sorted(k for k, v in ch_days.items() if v < 5),
      "| counting days over all 55 sessions instead:", sorted(k for k, v in icms.channel_days(sessions_all).items() if v < 5))
print("responses by current:", R.current.value_counts().sort_index().to_dict())
print("z range (mod50):", R.loc[R.mod50, "z"].min().round(2), "to", R.loc[R.mod50, "z"].max().round(2),
      "| time to max range (ms):", R.loc[R.mod50, "tmax_ms"].min(), "to", R.loc[R.mod50, "tmax_ms"].max())
print("responses with 5 trials or fewer:", int((R.n_trials <= 5).sum()), "| z >= 50 among Methods-modulated:",
      int((R.mod_methods & (R.z >= 50)).sum()))
print("PL among mod50 at 4-6 uA:", int(R.loc[R.mod50 & R.current.isin(icms.CURRENTS), "pl"].sum()), "of",
      int((R.mod50 & R.current.isin(icms.CURRENTS)).sum()))
print("modulated (mod50, 4-6 uA) by mouse and phase:")
print(pd.crosstab(R.loc[R.mod50 & R.current.isin(icms.CURRENTS), "mouse"], R.loc[R.mod50 & R.current.isin(icms.CURRENTS), "phase"]))

In [ ]:
# The pulse-locking call: how it treats synthetic responses, and where the PL calls peak
rng_syn = np.random.default_rng(50)


def synth_pulses(shape, rate, n=2100):
    """n pulses (30 trains of 70) with Poisson spikes in the unblanked 1.5 to 9.5 ms, shaped by `shape`."""
    c = (np.arange(20) + 0.5) * 0.5
    valid = (c >= 1.5) & (c <= 9.5)
    w = np.zeros(20)
    w[valid] = shape(c[valid])
    w /= w.sum()
    tot = rng_syn.poisson(rate * n)
    M = np.zeros((n, 20), np.int32)
    np.add.at(M, (rng_syn.integers(0, n, tot), rng_syn.choice(20, tot, p=w)), 1)
    return M


shapes = {"uniform (no locking)": lambda c: np.ones_like(c), "monotone ramp, no peak": lambda c: c,
          "peak at 4 ms (sd 0.5 ms)": lambda c: 1 + 3 * np.exp(-0.5 * ((c - 4) / 0.5) ** 2)}
syn = []
for name, f in shapes.items():
    for rate in [0.02, 0.1, 0.3]:
        calls = [icms.pulse_locking(synth_pulses(f, rate), rng_syn)["pl"] for _ in range(200)]
        syn.append({"spike train": name, "spikes per pulse": rate, "fraction called PL": np.mean(calls)})
print("fraction of 200 synthetic responses called PL (nominal false-positive rate of the call: 1%):")
print(pd.DataFrame(syn).pivot(index="spike train", columns="spikes per pulse", values="fraction called PL"))
Q = R[R.mod50 & R.current.isin(icms.CURRENTS) & R.phase.notna()].copy()
Q["spikes_per_pulse"] = Q.n_pl_spikes / (Q.n_trials * 70)
print("\nmodulated responses at 4-6 uA, unblanked spikes per pulse: quartiles",
      Q.spikes_per_pulse.quantile([0.25, 0.5, 0.75]).round(3).tolist())
print(f"PL: {int(Q.pl.sum())} of {len(Q)} ({Q.pl.mean():.3f})")
print("PL calls by the time of the peak of the mean curve (ms; 1.5 is the inserted point, 9.25 the last unblanked bin):")
print(Q[Q.pl.astype(bool)].peak_ms.value_counts().sort_index().to_dict())
late_peak = Q.pl.astype(bool) & (Q.peak_ms >= 8.75)
print(f"PL calls peaking in the last two unblanked bins (8.75, 9.25 ms): {int(late_peak.sum())}; "
      f"PL fraction without them: {(Q.pl.astype(bool) & ~late_peak).sum() / len(Q):.3f}")
print("PL latency (mean bootstrap peak time) range:", Q[Q.pl.astype(bool)].latency_ms.min().round(2), "to",
      Q[Q.pl.astype(bool)].latency_ms.max().round(2), "ms")
R["pl_strict"] = R.pl.fillna(False).astype(bool) & (R.peak_ms < 8.75)  # variant: PL calls that do not peak at the end

The call is conservative on uniform spikes: none of 600 synthetic responses without locking is called PL, below the nominal 1%. It still calls a monotone ramp PL when spikes are dense (0.705 of ramps at 0.3 spikes per pulse), but only 7 of the 1413 PL calls on the data peak in the last two unblanked bins, and without them the PL fraction is 0.620 instead of 0.623. The PL calls peak mostly at 2.75 to 4.75 ms. Round 1 built the null from the spikes of all pulses, which narrowed it and called 80% of responses PL; the paper's n imply about 62%.

## fig4-unit-counts

> Electrophysiology recorded 973 units in six animals (672 putative pyramidal, 301 interneurons),
> of which 484 were significantly modulated by the stimulation train (311 pyramidal, 173
> interneurons).

Units are counted per session (reading a); the files carry no identity of a unit across
sessions, so reading (b), unique units per animal, cannot be formed. Cell type is the stored
`cell_type` (ICMS83 uses other spellings, which are the same labels), with the trough-to-peak of
the stored mean waveform as a variant.

In [ ]:
def unit_counts(sel_sessions, mod_col, currents):
    rows = []
    for si, s in enumerate(sessions):
        if s not in sel_sessions:
            continue
        sub = R[(R.si == si) & R.current.isin(currents)]
        modu = set(sub.loc[sub[mod_col], "unit"])
        for u in range(len(s["spikes"])):
            ct_ = {"pyramidal": "pyramidal", "Pyramidal Cell": "pyramidal", "interneuron": "interneuron",
                   "Narrow Interneuron": "interneuron"}.get(str(s["cell_type"][u]), "unknown")
            rows.append({"mouse": icms.MOUSE[s["subject"]], "ctype": ct_, "mod": u in modu,
                         "accepted": bool(s["accepted"][u]),
                         "t2p": "interneuron" if trough_to_peak_ms(s["waveform"][u]) < 0.45 else "pyramidal"})
    return pd.DataFrame(rows)

uc_main = None
for label, sel, mod_col, cur in [("weeks 0-4, mod50, 4-6 uA (main)", sessions, "mod50", icms.CURRENTS),
                                 ("weeks 0-4, Methods filters, 4-6 uA", sessions, "mod_methods", icms.CURRENTS),
                                 ("weeks 0-4, mod50, all currents", sessions, "mod50", tuple(sorted(R.current.unique())))]:
    U = unit_counts(sel, mod_col, cur)
    if uc_main is None:
        uc_main = U
    print(f"{label}: units {len(U)}, pyramidal {int((U.ctype == 'pyramidal').sum())}, interneurons "
          f"{int((U.ctype == 'interneuron').sum())}, unknown {int((U.ctype == 'unknown').sum())}; modulated "
          f"{int(U['mod'].sum())} (pyramidal {int((U['mod'] & (U.ctype == 'pyramidal')).sum())}, interneurons "
          f"{int((U['mod'] & (U.ctype == 'interneuron')).sum())})")
U = uc_main
print("variant, cell type by the trough-to-peak rule on the stored mean waveform: all", U.t2p.value_counts().to_dict(),
      "| modulated", U.loc[U["mod"], "t2p"].value_counts().to_dict())
UA_ = U[U.accepted]
print(f"variant, accepted units only: units {len(UA_)}, pyramidal {int((UA_.ctype == 'pyramidal').sum())}, interneurons "
      f"{int((UA_.ctype == 'interneuron').sum())}; modulated {int(UA_['mod'].sum())}")
# variant: all 55 sessions, weeks 5 and 6 included
extra = [s for s in sessions_all if s["week"] > 4]
rx = icms.response_table(extra, rng_seed=5, currents=icms.CURRENTS)
ch_days_all = icms.channel_days(sessions_all)
mod_x = {(id(extra[r["si"]]), r["unit"]) for r in rx if icms.modulated(r, 50, True, ch_days_all)}
ux = pd.DataFrame([{"ctype": {"pyramidal": "pyramidal", "Pyramidal Cell": "pyramidal", "interneuron": "interneuron",
                              "Narrow Interneuron": "interneuron"}.get(str(s["cell_type"][u]), "unknown"),
                    "mod": (id(s), u) in mod_x} for s in extra for u in range(len(s["spikes"]))])
UA = pd.concat([U[["ctype", "mod"]], ux])
print(f"variant, all 55 sessions: units {len(UA)}, pyramidal {int((UA.ctype == 'pyramidal').sum())}, interneurons "
      f"{int((UA.ctype == 'interneuron').sum())}; modulated {int(UA['mod'].sum())} (pyramidal "
      f"{int((UA['mod'] & (UA.ctype == 'pyramidal')).sum())}, interneurons {int((UA['mod'] & (UA.ctype == 'interneuron')).sum())})")
print("per mouse:", U.groupby("mouse").size().to_dict())
t2p_all = np.array([trough_to_peak_ms(w) for s in sessions for w in s["waveform"]])
h, e = np.histogram(t2p_all, bins=np.arange(0, 1.2, 0.05))
print("trough-to-peak histogram (0.05 ms bins from 0):", h.tolist())
# trough-to-peak takes whole samples at 30 kHz (1/30 ms), so a 0.05 ms bin holds one or two possible values
n_vals, _ = np.histogram(np.arange(40) / 30000.0 * 1000.0, bins=e)
print("possible values per bin:", n_vals.tolist())
print("units per possible value, by bin (count / values):",
      [round(a / b, 1) if b else None for a, b in zip(h.tolist(), n_vals.tolist())])
_v, _c = np.unique(np.round(t2p_all, 4), return_counts=True)
print("units at each value from 0.30 to 0.70 ms:", dict(zip(np.round(_v[(_v >= 0.3) & (_v <= 0.7)], 3).tolist(),
                                                           _c[(_v >= 0.3) & (_v <= 0.7)].tolist())))

**Result: not reproduced.** In weeks 0 to 4 the files hold 795 unit-sessions (584 pyramidal, 204 interneurons, 7 unknown), of which 488 are modulated (339 pyramidal, 146 interneurons): the modulated total and the modulated pyramidal count are within tolerance, the other four counts are below it, 2 of 6 where the criterion asks 5. Over all 55 sessions there are 999 units (732 and 260) and 585 modulated (405 and 177), 3 of 6. With the trough-to-peak rule applied to the stored mean waveforms, 577 pyramidal and 218 interneurons, 334 and 154 modulated. Every unit is marked accepted, so restricting to accepted units changes nothing. Reading (b) cannot be formed. The trough-to-peak histogram in 0.05 ms bins alternates (59, 22, 44 at 0.40 to 0.55 ms; 128, 61, 150 at 0.55 to 0.70 ms), but at 30 kHz a bin holds one or two possible values, and per value the counts at 0.40 to 0.55 ms are 29.5, 22 and 22, so the dip in the 0.45 to 0.50 ms bin comes from the binning. No bimodality is judged.

## figs3b-modulation-increase-all

> The modulation z-score of all modulated responses at fixed currents increases from the early to
> the late phase at 4, 5 and 6 uA.

In [ ]:
def phase_test(df, col, label, alternative="two-sided"):
    out = {}
    for c in icms.CURRENTS:
        d = df[df.current == c]
        t = icms.mwu(d.loc[d.phase == "early", col], d.loc[d.phase == "late", col], alternative)
        out[c] = t
        print(f"{label} {int(c)} uA: early median {t['med1']:.3f} (n={t['n1']}), late median {t['med2']:.3f} "
              f"(n={t['n2']}), p = {t['p']:.3g}, r = {t['r']:.2f}")
    return out

M50 = R[R.mod50 & R.current.isin(icms.CURRENTS) & R.phase.notna()]
res_s3b = phase_test(M50, "z", "z")
print("variant, Methods filters only:")
_ = phase_test(R[R.mod_methods & R.current.isin(icms.CURRENTS) & R.phase.notna()], "z", "z")
print("by mouse (median z, early / late, all three currents):")
print(M50.pivot_table(index="mouse", columns="phase", values="z", aggfunc="median").round(2))

**Result: reproduced.** The modulation z-score of modulated responses is higher late than early at 4 uA (2.706 against 3.765, p = 9.57e-05, r = 0.18), 5 uA (3.399 against 3.875, p = 0.00656, r = 0.12) and 6 uA (3.559 against 4.570, p = 0.000404, r = 0.17). The response counts (191/738, 284/498, 300/256) are close to the paper's (189/729, 283/489, 303/250).

## figs3c-latency-all

> Time to maximum firing rate of all modulated responses decreases at 5 uA but not significantly at
> 4 or 6 uA.

In [ ]:
res_s3c = phase_test(M50, "tmax_ms", "time to max (ms)")

**Result: partially reproduced.** Time to maximum falls at 5 uA (80 to 45 ms, p = 0.00522) and does not change significantly at 6 uA (p = 0.168), as the paper reports, but it also falls significantly at 4 uA (75 to 55 ms, p = 0.0465, r = -0.09), where the paper reports p = 0.18.

## figs3d-modulation-stable-at-threshold

> Modulation at each session's detection threshold does not change over training.

The session threshold of a channel (psignifit, fig2a) rounded to the nearest uA picks the current.

In [ ]:
thr_cur = {(r.subject, r.date, r.channel): int(np.round(r.ps)) for r in thr.itertuples()}
R["thr_cur"] = [thr_cur.get((a, b, c), -1) for a, b, c in zip(R.subject, R.date, R.channel)]
AT = R[R.mod50 & (R.current == R.thr_cur) & R.phase.notna()]
per_cs = AT.groupby(["subject", "date", "channel", "phase"], as_index=False).z.median()
ta = icms.mwu(per_cs.loc[per_cs.phase == "early", "z"], per_cs.loc[per_cs.phase == "late", "z"])
tb = icms.mwu(AT.loc[AT.phase == "early", "z"], AT.loc[AT.phase == "late", "z"])
print("threshold currents used:", AT.current.value_counts().sort_index().to_dict())
print(f"(a) channel-session medians: early {ta['med1']:.2f} (n={ta['n1']}), late {ta['med2']:.2f} (n={ta['n2']}), "
      f"p = {ta['p']:.3g}, r = {ta['r']:.2f}")
print(f"(b) all responses: early {tb['med1']:.2f} (n={tb['n1']}), late {tb['med2']:.2f} (n={tb['n2']}), "
      f"p = {tb['p']:.3g}, r = {tb['r']:.2f}")
# x50 threshold reading
thr_cur2 = {(r.subject, r.date, r.channel): int(np.round(r.x50)) if not np.isnan(r.x50) else -1 for r in thr.itertuples()}
AT2 = R[R.mod50 & (R.current == [thr_cur2.get((a, b, c), -1) for a, b, c in zip(R.subject, R.date, R.channel)]) & R.phase.notna()]
p2 = AT2.groupby(["subject", "date", "channel", "phase"], as_index=False).z.median()
t2 = icms.mwu(p2.loc[p2.phase == "early", "z"], p2.loc[p2.phase == "late", "z"])
print(f"(a) with the MLE x50 threshold: early n={t2['n1']}, late n={t2['n2']}, p = {t2['p']:.3g}, r = {t2['r']:.2f}")

**Result: reproduced.** At the current nearest each session's psignifit threshold, the channel-session median z-scores do not differ between phases (reading a: early 3.93, n = 42; late 3.30, n = 87; p = 0.16, r = -0.15), nor do all modulated responses (reading b, p = 0.789); with the maximum likelihood x50 threshold, p = 0.702. The n of reading (a) are about twice the paper's 21 and 34, so the criterion's choice of (a) as the reading whose n matches does not hold here.

## fig4g-modulated-count-mouse1

> The number of modulated units rose moderately in some animals (Mouse 1 at 4 uA), with no
> significant change at group level.

In [ ]:
cnt = (R[(R.current == 4) & R.phase.notna()].groupby(["subject", "mouse", "date", "channel", "phase"], as_index=False)
       .mod50.sum().rename(columns={"mod50": "n_mod"}))
m1 = cnt[cnt.subject == "ICMS92"]
t = icms.mwu(m1.loc[m1.phase == "early", "n_mod"], m1.loc[m1.phase == "late", "n_mod"], "less")
print(f"Mouse 1, 4 uA: early median {t['med1']} (n={t['n1']}), late median {t['med2']} (n={t['n2']}), "
      f"one-sided p = {t['p']:.3g}, r = {t['r']:.2f}")
res_fig4g = t
tg = icms.mwu(cnt.loc[cnt.phase == "early", "n_mod"], cnt.loc[cnt.phase == "late", "n_mod"], "less")
print(f"all six mice, 4 uA: early median {tg['med1']} (n={tg['n1']}), late median {tg['med2']} (n={tg['n2']}), "
      f"one-sided p = {tg['p']:.3g}, r = {tg['r']:.2f}")
for sub in icms.BEHAVIOUR:
    d = cnt[cnt.subject == sub]
    tt = icms.mwu(d.loc[d.phase == "early", "n_mod"], d.loc[d.phase == "late", "n_mod"], "less")
    print(f"  {icms.MOUSE[sub]}: early mean {d.loc[d.phase == 'early', 'n_mod'].mean():.2f} (n={tt['n1']}), "
          f"late mean {d.loc[d.phase == 'late', 'n_mod'].mean():.2f} (n={tt['n2']}), one-sided p = {tt['p']:.3g}")

**Result: reproduced.** In Mouse 1 at 4 uA the number of modulated responses per channel and session is higher late than early (median 12 against 14, n = 8 and 21 as in the paper, one-sided p = 0.0443, r = 0.42). Over all six mice the same test is not significant (p = 0.0728).

## fig4ij-pyramidal-modulation-increase

> Increases in modulation over training are carried by pyramidal cells, at 4, 5 and 6 uA, with
> latency decreasing at 5 and 6 uA.

z cap 100 (claims). The code's one-sided tests are a variant.

In [ ]:
M100 = R[R.mod100 & R.current.isin(icms.CURRENTS) & R.phase.notna()]
PYR = M100[M100.ctype == "pyramidal"]
res_pyr_z = phase_test(PYR, "z", "pyramidal z")
res_pyr_t = phase_test(PYR, "tmax_ms", "pyramidal time to max (ms)")
print("variant, one-sided (late greater for z, late smaller for time to max):")
_ = phase_test(PYR, "z", "pyramidal z", "less")
_ = phase_test(PYR, "tmax_ms", "pyramidal time to max", "greater")
print("variant, cell type by the trough-to-peak rule on the stored mean waveform:")
_ = phase_test(M100[M100.t2p_type == "pyramidal"], "z", "pyramidal (t2p) z")
_ = phase_test(M100[M100.t2p_type == "pyramidal"], "tmax_ms", "pyramidal (t2p) time to max")

**Result: reproduced.** Pyramidal modulation rises at 4, 5 and 6 uA (p = 4.79e-05, 0.000587, 0.000878) and time to maximum falls at 5 uA (p = 0.000329) and 6 uA (p = 0.025). It also falls at 4 uA (p = 0.0208), where the paper reports p = 0.32. With cell types from the trough-to-peak rule on the stored mean waveform, modulation rises at all three currents (p = 0.000212, 0.000435, 0.000492) and time to maximum falls at 5 uA (p = 0.000362) but not significantly at 6 uA (p = 0.0587).

## fig4ij-interneurons-little-change

> Interneurons show no significant change in modulation at any current; their latency changes
> significantly only at 6 uA.

In [ ]:
INT = M100[M100.ctype == "interneuron"]
res_int_z = phase_test(INT, "z", "interneuron z")
res_int_t = phase_test(INT, "tmax_ms", "interneuron time to max (ms)")
print("interneuron responses by mouse:", INT.mouse.value_counts().to_dict())
print("variant, cell type by the trough-to-peak rule on the stored mean waveform:")
_ = phase_test(M100[M100.t2p_type == "interneuron"], "z", "interneuron (t2p) z")
_ = phase_test(M100[M100.t2p_type == "interneuron"], "tmax_ms", "interneuron (t2p) time to max")

**Result: reproduced.** Interneuron modulation does not change at 4, 5 or 6 uA (p = 0.331, 0.884, 0.0885; r = 0.08, -0.01, 0.15). Their time to maximum changes only at 6 uA (130 to 190 ms, p = 0.0335, r = 0.19; paper p = 0.010, r = 0.21), not at 4 or 5 uA (p = 0.929, 0.361). With trough-to-peak cell types, modulation does not change either (p = 0.117, 0.706, 0.229), and time to maximum changes at no current (6 uA p = 0.419).

## fig5d-pulse-locking-stable-across-currents

> Over 80% of units keep their PL or NPL identity at a given stimulation contact as the current
> rises from 4 to 6 uA.

In [ ]:
def identity_patterns(df):
    rows_ = []
    for (si, ch, u), g in df.groupby(["si", "channel", "unit"]):
        if len(g) < 2:
            continue
        seq = g.sort_values("current").pl.astype(bool).tolist()
        p = ("all PL" if all(seq) else "all NPL" if not any(seq) else "NPL to PL" if seq == sorted(seq)
             else "PL to NPL" if seq == sorted(seq, reverse=True) else "mixed")
        rows_.append({"mouse": g.mouse.iloc[0], "pattern": p, "n_currents": len(g)})
    return pd.DataFrame(rows_)


pat_rows = []
for (si, ch, u), g in M50.groupby(["si", "channel", "unit"]):
    if len(g) < 2:
        continue
    seq = g.sort_values("current").pl.astype(bool).tolist()
    if all(seq):
        p = "all PL"
    elif not any(seq):
        p = "all NPL"
    elif seq == sorted(seq):
        p = "NPL to PL"
    elif seq == sorted(seq, reverse=True):
        p = "PL to NPL"
    else:
        p = "mixed"
    pat_rows.append({"mouse": g.mouse.iloc[0], "pattern": p, "n_currents": len(g)})
PAT = pd.DataFrame(pat_rows)
vc = PAT.pattern.value_counts()
frac_keep = (vc.get("all PL", 0) + vc.get("all NPL", 0)) / len(PAT)
print("unit-channel pairs seen at two or more currents:", len(PAT), "| patterns:", vc.to_dict())
print(f"fraction keeping PL or NPL identity: {frac_keep:.3f}")
print("by mouse:", PAT.groupby("mouse").pattern.apply(lambda x: round(x.isin(["all PL", "all NPL"]).mean(), 3)).to_dict())
# The PL call is random (the bootstrap draws and the null). The same responses called again with three
# other seeds, for this claim and for fig5e and fig5h:
pl_reseeded = {}
t0 = time.time()
for seed in [11, 12, 13]:
    rng_s = np.random.default_rng(seed)
    labs = []
    for r in M50.itertuples():
        s = sessions[r.si]
        tr = icms.ephys_trains(s)
        m = (tr["stim_channel"] == r.channel) & (tr["current_uA"] == r.current)
        Mx = icms.pulse_matrix(s["spikes"][r.unit], icms.pulse_starts(np.sort(tr["start_time"][m])))
        labs.append(icms.pulse_locking(Mx, rng_s)["pl"])
    pl_reseeded[seed] = np.array(labs)
    P2 = identity_patterns(M50.assign(pl=pl_reseeded[seed]))
    print(f"seed {seed}: PL {int(pl_reseeded[seed].sum())} of {len(M50)}, pairs {len(P2)}, fraction keeping identity "
          f"{P2.pattern.isin(['all PL', 'all NPL']).mean():.3f}")
print(f"(in {time.time() - t0:.0f} s)")

**Result: not reproduced.** Of 740 unit-channel pairs seen at two or more currents (paper 729), 389 stay PL and 197 stay NPL, a fraction of 0.792, below the 0.80 the claim states; by mouse it ranges from 0.72 to 0.908. Called again with three other seeds, the fraction is 0.774, 0.793 and 0.789, below 0.80 each time.

## fig5e-pyramidal-fraction-pl

> Pyramidal cells are more common among PL units (76.2%) than among NPL units (49.8%).

Two-step majority: over stimulation channels at each current, then over currents, ties NPL.
Units of unknown type are left out of the fractions.

In [ ]:
def unit_classes(df):
    out_ = []
    for (si, u), g in df.groupby(["si", "unit"]):
        per_cur = []
        for c, gc in g.groupby("current"):
            k = int(gc.pl.astype(bool).sum())
            per_cur.append(k > len(gc) - k)
        k = sum(per_cur)
        out_.append({"si": si, "unit": u, "ctype": g.ctype.iloc[0], "PL": k > len(per_cur) - k})
    return pd.DataFrame(out_)


ucls = []
for (si, u), g in M50.groupby(["si", "unit"]):
    per_cur = []
    for c, gc in g.groupby("current"):
        k = int(gc.pl.astype(bool).sum())
        per_cur.append(k > len(gc) - k)
    k = sum(per_cur)
    ucls.append({"si": si, "unit": u, "ctype": g.ctype.iloc[0], "mouse": g.mouse.iloc[0],
                 "PL": k > len(per_cur) - k})
UC = pd.DataFrame(ucls)
tab5e = pd.crosstab(UC.PL, UC.ctype)
print(tab5e)
known = UC[UC.ctype != "unknown"]
f_pl = (known[known.PL].ctype == "pyramidal").mean()
f_npl = (known[~known.PL].ctype == "pyramidal").mean()
odds, p_f = stats.fisher_exact(pd.crosstab(known.PL, known.ctype).values)
print(f"pyramidal fraction: PL units {f_pl:.3f} (n={int(known.PL.sum())}), NPL units {f_npl:.3f} "
      f"(n={int((~known.PL).sum())}); Fisher exact p = {p_f:.3g}, odds ratio {odds:.2f}")
# variant: cell type from the trough-to-peak of the stored waveform
t2p_of = {(si, u): ("interneuron" if trough_to_peak_ms(sessions[si]["waveform"][u]) < 0.45 else "pyramidal")
          for si, u in zip(UC.si, UC.unit)}
UC["t2p"] = [t2p_of[(a, b)] for a, b in zip(UC.si, UC.unit)]
fp2, fn2 = (UC[UC.PL].t2p == "pyramidal").mean(), (UC[~UC.PL].t2p == "pyramidal").mean()
print(f"variant, trough-to-peak types: PL {fp2:.3f}, NPL {fn2:.3f}, Fisher p = "
      f"{stats.fisher_exact(pd.crosstab(UC.PL, UC.t2p).values)[1]:.3g}")
for seed, labs in pl_reseeded.items():
    U2 = unit_classes(M50.assign(pl=labs))
    U2 = U2[U2.ctype != "unknown"]
    print(f"seed {seed}: pyramidal fraction PL {(U2[U2.PL].ctype == 'pyramidal').mean():.3f} (n={int(U2.PL.sum())}), "
          f"NPL {(U2[~U2.PL].ctype == 'pyramidal').mean():.3f} (n={int((~U2.PL).sum())}), Fisher p = "
          f"{stats.fisher_exact(pd.crosstab(U2.PL, U2.ctype).values)[1]:.3g}")

**Result: partially reproduced.** Pyramidal cells are 0.815 of PL units (n = 270), within tolerance, and 0.553 of NPL units (n = 215), 0.005 above the tolerance of 0.448 to 0.548; PL units are more often pyramidal (Fisher exact p = 7.73e-10, odds ratio 3.55). With three other seeds of the PL call, the NPL fraction is 0.565, 0.559 and 0.548, and the PL fraction 0.813 to 0.826. With cell types from the trough-to-peak rule on the stored mean waveform, 0.791 and 0.549 (p = 1.67e-08).

## figs4b-pl-latency-distance

> PL spike latency increases linearly with distance from the stimulation site, giving conduction
> velocities of about 0.2 to 0.27 m/s; latency jitter does not vary with distance.

Distance follows the paper's code (processing/batch_process/postprocessing/longitudinal_data_utils.py,
add_data_to_df and correct_df_unit_location; python/fig_s4/generate_figure.py): the stimulation
contact sits at (32 - stim_channel) x 60 um up the probe, the unit at its estimated position
(`unit_y_um`); both are turned into cortical depth, (1860 - height - the animal's length of probe
outside the brain) x cos 30 degrees, responses with either depth at or above the surface are left
out, and the distance is the difference in depth. The code has these lengths for Mice 1 to 5 only, so
Mouse 6 is left out, as in the code. PL responses are the modulated ones with the z cap of 100 that
the code's figure uses (filter_modulated(max_z_score=100)). Latency readings over responses at 120 to
840 um, in 120 um bins from 60 um: (a) binned means (legend), (b) binned medians (code); jitter (c)
binned means and (d) unbinned responses.

Round 1 placed the contact at the `rel_y` of the electrode whose `channel_name` equals
`stim_channel`; that mapping is kept as a variant. The two disagree: the electrode table puts
channel 32 at 1860 um and the code puts stimulation channel 32 at 0 um. A check of which is right
is printed first: the depth of each animal's stimulation contacts against the Methods' 200 to 400 um,
and the fraction of modulated responses against distance under each mapping.

In [ ]:
# lengths of probe outside the brain (um), from the paper's code (correct_df_unit_location)
OUTSIDE_UM = {"ICMS92": 300, "ICMS93": 180, "ICMS98": 60, "ICMS100": -300, "ICMS101": 360}


def cortical_depth(height_um, subject):
    return (1860.0 - height_um - OUTSIDE_UM[subject]) * np.cos(np.deg2rad(30))


def stim_height_code(ch):
    return (32 - ch) * 60.0


def stim_height_table(si, ch):
    s = sessions[si]
    k = np.flatnonzero(s["el_channel"] == ch)
    return s["el_y"][k[0]] if len(k) else np.nan


# check of the mapping 1: depth of the stimulation contacts (Methods: 200 to 400 um)
dep = []
for sub in OUTSIDE_UM:
    si0 = [i for i, s in enumerate(sessions) if s["subject"] == sub]
    chs = sorted(set(R.loc[R.si.isin(si0) & R.current.isin(icms.CURRENTS), "channel"]))
    for ch in chs:
        dep.append({"mouse": icms.MOUSE[sub], "stim_channel": ch,
                    "depth_code_um": float(np.round(cortical_depth(stim_height_code(ch), sub))),
                    "depth_electrode_table_um": float(np.round(cortical_depth(stim_height_table(si0[0], ch), sub)))})
print(pd.DataFrame(dep).to_string())

# check of the mapping 2: modulated fraction against distance, all responses at 4-6 uA
A = R[R.current.isin(icms.CURRENTS) & R.phase.notna()].copy()
A["dist_code"] = [abs(sessions[si]["unit_y"][u] - stim_height_code(ch)) for si, u, ch in zip(A.si, A.unit, A.channel)]
A["dist_table"] = [abs(sessions[si]["unit_y"][u] - stim_height_table(si, ch)) for si, u, ch in zip(A.si, A.unit, A.channel)]
for col in ["dist_code", "dist_table"]:
    print(f"\nall responses at 4-6 uA by distance along the probe, {col} (um):")
    print(A.groupby(pd.cut(A[col], [0, 60, 120, 240, 480, 960, 2000]), observed=False).agg(
        fraction_modulated=("mod50", "mean"), n=("mod50", "size"), median_z=("z", "median")).round(3).T)
    print("  fraction modulated within 240 um and beyond 960 um, by mouse:",
          {m: (round(g.loc[g[col] <= 240, "mod50"].mean(), 2), round(g.loc[g[col] > 960, "mod50"].mean(), 2))
           for m, g in A.groupby("mouse")})


print("\nfraction of modulated responses by distance along the probe (code mapping, um), by mouse:")
_g = A.groupby(["mouse", pd.cut(A.dist_code, [0, 60, 120, 240, 480, 960, 2000])], observed=False).mod50
print(pd.concat({"fraction": _g.mean().round(2), "n": _g.size()}, axis=1).unstack(1).to_string())


def s4b_fits(PLR, label, verbose=True):
    edges = np.arange(60, 901, 120)
    centres = (edges[:-1] + edges[1:]) / 2
    out = {}
    for c in icms.CURRENTS:
        d = PLR[(PLR.current == c) & PLR.dist.notna()]
        dr = d[(d.dist >= 120) & (d.dist <= 840)]
        b = np.digitize(dr.dist, edges) - 1
        o = {}
        for agg in ["mean", "median"]:
            y = np.array([getattr(np, agg)(dr.latency_ms[b == i]) if (b == i).sum() else np.nan for i in range(len(centres))])
            use = ~np.isnan(y)
            lr = stats.linregress(centres[use], y[use])
            o[agg] = (lr.slope, lr.pvalue, lr.rvalue ** 2, 1e-3 / lr.slope if lr.slope > 0 else np.nan)
            if verbose:
                print(f"{label} {int(c)} uA latency, binned {agg}s 120-840 um: slope {lr.slope:.5f} ms/um, p = {lr.pvalue:.3g}, "
                  f"r2 = {lr.rvalue ** 2:.2f}, velocity {o[agg][3]:.3f} m/s; bin means/medians "
                      f"{np.round(y, 2).tolist()}, n {[int((b == i).sum()) for i in range(len(centres))]}")
        bj = np.digitize(d.dist, edges) - 1
        yj = np.array([np.mean(d.jitter_ms2[bj == i]) if (bj == i).sum() else np.nan for i in range(len(centres))])
        use = ~np.isnan(yj)
        lj = stats.linregress(centres[use], yj[use])
        lj2 = stats.linregress(d.dist, d.jitter_ms2)
        o["jitter_binned"], o["jitter_unbinned"] = (lj.slope, lj.pvalue), (lj2.slope, lj2.pvalue)
        if verbose:
            print(f"   jitter: binned means slope {lj.slope:.2e} ms2/um p = {lj.pvalue:.3g}; unbinned slope {lj2.slope:.2e} "
                  f"p = {lj2.pvalue:.3g}; PL responses {len(d)}, at 120-840 um {len(dr)}; latency range "
                  f"{d.latency_ms.min():.2f} to {d.latency_ms.max():.2f} ms")
        out[c] = o
    return out


P100 = M100[M100.pl.astype(bool) & (M100.subject != "ICMS83")].copy()
P100["d_unit"] = [cortical_depth(sessions[si]["unit_y"][u], sub) for si, u, sub in zip(P100.si, P100.unit, P100.subject)]
P100["d_stim"] = [cortical_depth(stim_height_code(ch), sub) for ch, sub in zip(P100.channel, P100.subject)]
inside = (P100.d_unit > 0) & (P100.d_stim > 0)
print(f"\nPL responses (z cap 100, Mice 1 to 5): {len(P100)}; left out with a depth at or above the surface: {int((~inside).sum())}")
PLR = P100[inside].copy()
PLR["dist"] = (PLR.d_unit - PLR.d_stim).abs()
print("distance range", PLR.dist.min().round(1), "to", PLR.dist.max().round(1), "um")
res_s4b = s4b_fits(PLR, "code mapping")
print("\nvariant: electrode-table mapping (round 1), distance along the probe, PL responses of all six mice")
PLT = M100[M100.pl.astype(bool)].copy()
PLT["dist"] = [abs(sessions[si]["unit_y"][u] - stim_height_table(si, ch)) for si, u, ch in zip(PLT.si, PLT.unit, PLT.channel)]
print("PL responses without a distance (stimulation contact not in the electrode table):", int(PLT.dist.isna().sum()))
res_s4b_table = s4b_fits(PLT, "table mapping")
print("\nvariant: code mapping without Mouse 5, whose modulated fraction does not fall with distance")
res_s4b_no5 = s4b_fits(PLR[PLR.subject != "ICMS101"], "code mapping, Mice 1 to 4")
print("\ncode mapping, leaving out one mouse at a time: velocity (m/s) from binned means / binned medians")
for sub in OUTSIDE_UM:
    o_ = s4b_fits(PLR[PLR.subject != sub], "", verbose=False)
    print(f"  without {icms.MOUSE[sub]} (PL responses {int((PLR.subject != sub).sum())}): " + "; ".join(
        f"{int(c)} uA {o_[c]['mean'][3]:.3f} / {o_[c]['median'][3]:.3f} (p {o_[c]['mean'][1]:.2g} / {o_[c]['median'][1]:.2g})"
        for c in icms.CURRENTS))
print("PL responses per mouse at 120-840 um:", PLR[(PLR.dist >= 120) & (PLR.dist <= 840)].groupby("mouse").size().to_dict())


**Result: partially reproduced.** With the stimulation contact placed as the paper's code places it, PL latency rises linearly with distance at every current (binned means over 120 to 840 um: slopes 0.00591, 0.00445, 0.00442 ms/um, p = 0.000384, 0.000314, 0.00079, r2 0.91 to 0.94; binned medians: p = 0.000147, 0.000356, 0.000486), and jitter shows no slope (binned p = 0.148, 0.33, 0.377; unbinned 0.79, 0.819, 0.143). The velocities are 0.169, 0.225 and 0.226 m/s from binned means and 0.161, 0.188 and 0.199 m/s from binned medians: at 4 uA both are below the 0.18 to 0.297 m/s the criterion allows, so no reading has all three inside. Pooled over mice, the outputs favour the code's mapping: the contacts lie at 0 to 520 um depth, 12 of 17 between 150 and 420 um (Methods: 200 to 400 um), against -156 to 1559 um with the electrode table's `rel_y`, and the fraction of modulated responses falls from 0.711 at 60 to 120 um to 0.390 beyond 960 um, where under the electrode-table mapping it does not fall (0.299 to 0.513). The check is not clean: the 0 to 60 um bin is the lowest (0.315), and per mouse the fall reverses in Mouse 5 (0.31 within 240 um, 0.48 beyond 960 um) and Mouse 6 (0.08 and 0.88; Mouse 6 is not in the fit), while Mouse 5 looks local under the electrode-table mapping (0.68 and 0.56). Without Mouse 5 the 4 uA velocity is 0.179 m/s from both binned means and medians (5 and 6 uA 0.190 to 0.229), still below 0.18; leaving out one mouse at a time, the 4 uA velocity is 0.146 to 0.189 m/s, and reaches 0.18 under both aggregations only without Mouse 1 (0.189 and 0.180). Under the electrode-table mapping no latency slope is significant (p 0.294 to 0.962).

## fig5fg-pl-modulation-increase

> PL modulation over the 700 ms train increases from early to late at 4, 5 and 6 uA, and PL time to
> maximum firing rate decreases at 5 and 6 uA.

In [ ]:
PLm = M50[M50.pl.astype(bool)]
NPLm = M50[~M50.pl.astype(bool)]
res_pl_z = phase_test(PLm, "z", "PL z")
res_pl_t = phase_test(PLm, "tmax_ms", "PL time to max (ms)")
print("variant, PL calls peaking in the last two unblanked bins counted as NPL:")
PLs = M50[M50.pl_strict]
_ = phase_test(PLs, "z", "PL (strict) z")
_ = phase_test(PLs, "tmax_ms", "PL (strict) time to max (ms)")

**Result: reproduced.** PL modulation rises at 4, 5 and 6 uA (p = 2.64e-05, 0.000333, 0.000345; r = 0.24, 0.19, 0.23) and PL time to maximum falls at 5 uA (75 to 30 ms, p = 0.0031) and 6 uA (75 to 35 ms, p = 0.041), not significantly at 4 uA (p = 0.0858), as the paper reports. The PL n (137/468, 187/300, 180/141) are close to the paper's (136/469, 188/285, 179/135). Counting the PL calls that peak in the last two unblanked bins as NPL gives the same six results.

## figs4de-npl-no-change

> NPL responses show no significant change in modulation or time to maximum firing rate over
> training at any current.

In [ ]:
res_npl_z = phase_test(NPLm, "z", "NPL z")
res_npl_t = phase_test(NPLm, "tmax_ms", "NPL time to max (ms)")
print("variant, PL calls peaking in the last two unblanked bins counted as NPL:")
NPLs = M50[~M50.pl_strict]
_ = phase_test(NPLs, "z", "NPL (strict) z")
_ = phase_test(NPLs, "tmax_ms", "NPL (strict) time to max (ms)")

**Result: reproduced.** NPL modulation does not change at 4, 5 or 6 uA (p = 0.185, 0.756, 0.174; r = 0.11, 0.02, 0.10), nor does NPL time to maximum (p = 0.0824, 0.386, 0.942). The NPL n (54/270, 97/198, 120/115) are close to the paper's (53/260, 95/204, 124/115). The variant that counts late-peaking PL calls as NPL gives the same.

## fig5h-pl-spike-probability-increase

> PL responses show increased peak spike probability within the 10 ms interpulse interval over
> training at 5 and 6 uA, not at 4 uA.

In [ ]:
res_pl_pk = phase_test(PLm, "peak_prob", "PL peak spike probability")
print("peak probability range:", PLm.peak_prob.min().round(4), "to", PLm.peak_prob.max().round(4))
print("variant, PL calls peaking in the last two unblanked bins counted as NPL:")
_ = phase_test(PLs, "peak_prob", "PL (strict) peak spike probability")
print("the PL calls under three other seeds (see fig5d):")
for seed, labs in pl_reseeded.items():
    _ = phase_test(M50[labs], "peak_prob", f"seed {seed}, PL peak spike probability")

**Result: partially reproduced.** The PL peak spike probability rises at 6 uA (0.047 to 0.061, p = 0.0113, r = 0.16) but not significantly at 5 uA (0.045 to 0.051, p = 0.0613, r = 0.10); at 4 uA p = 0.105, as the paper reports no change there. The 5 uA test depends on the random PL call: with three other seeds it gives p = 0.0197, 0.0275 and 0.0485, and the variant that counts late-peaking PL calls as NPL gives p = 0.0499.

## figs5-restricted-window-consistent

> In a 120 ms window after stimulation onset, before movement, PL modulation still increases at
> all currents while NPL modulation does not change, and PL time to maximum falls at 4 and 5 uA.

Spikes in the first 120 ms of the train against the 120 ms before it; time to maximum in 2 ms
bins smoothed by a 5 ms Gaussian. Readings: (keep) modulated set and PL labels from the 700 ms
window; (recompute) modulated by the 120 ms test and PL from the first 12 pulses.

In [ ]:
t0 = time.time()
r120 = icms.response_table(sessions, rng_seed=3, currents=icms.CURRENTS, window=0.12, tmax_bin=0.002, tmax_sigma=0.005)
key700 = {(r["si"], r["unit"], r["channel"], r["current"]): r for r in resp_rows}
rng120 = np.random.default_rng(4)
for r in r120:
    r["mod120"] = icms.modulated(r, 50, True, ch_days)
    k = key700[(r["si"], r["unit"], r["channel"], r["current"])]
    r["mod700"], r["pl700"] = k["mod50"], bool(k.get("pl", False))
    if r["mod120"]:
        s = sessions[r["si"]]
        tr = icms.ephys_trains(s)
        m = (tr["stim_channel"] == r["channel"]) & (tr["current_uA"] == r["current"])
        Mx = icms.pulse_matrix(s["spikes"][r["unit"]], icms.pulse_starts(np.sort(tr["start_time"][m]), 12))
        r["pl120"] = icms.pulse_locking(Mx, rng120)["pl"]
R120 = pd.DataFrame(r120)
R120 = R120[R120.phase.notna()]
print(f"120 ms responses: {len(R120)} in {time.time() - t0:.0f} s; modulated in 120 ms: {int(R120.mod120.sum())}")
res_s5 = {}
for reading, sel, lab in [("keep", R120.mod700, "pl700"), ("recompute", R120.mod120, "pl120")]:
    d = R120[sel]
    pl = d[d[lab].astype(bool)]
    npl = d[~d[lab].astype(bool)]
    print(f"\nreading {reading}:")
    res_s5[reading] = {"pl_z": phase_test(pl, "z", "PL z"), "npl_z": phase_test(npl, "z", "NPL z"),
                       "pl_t": phase_test(pl, "tmax_ms", "PL time to max"),
                       "npl_t": phase_test(npl, "tmax_ms", "NPL time to max")}

**Result: reproduced, under the reading that keeps the 700 ms labels.** In the first 120 ms, PL modulation rises at 4, 5 and 6 uA (p = 0.000508, 0.000218, 0.000116), NPL modulation does not change (p = 0.254, 0.633, 0.776), and PL time to maximum falls at 4 and 5 uA (p = 0.0107, 0.0119) and not at 6 uA (p = 0.994), as the paper reports; NPL time to maximum does not change (p = 0.992, 0.964, 0.607). With the modulated set and labels recomputed in the 120 ms window, NPL modulation rises at all three currents (p = 0.0368, 0.000146, 5.56e-05) and PL modulation does not rise at 6 uA (p = 0.14).

## fig6c-kl-divergence

> Hit-miss separation of delta-spikes distributions (KL divergence) increases over time for both
> PL and NPL responses, and NPL responses separate hit from miss more than PL responses in both
> early and late phases.

Trials of the modulated responses (mod50) at 4 to 6 uA, pooled per week and label. Readings (k1)
winsorizing 1%, fraction 0.4 and (k2) 2.5%, fraction 0.8; 1000 draws.

In [ ]:
lab_of = {(r.si, r.unit, r.channel, r.current): bool(r.pl) for r in M50.itertuples()}
trial_pool = defaultdict(lambda: {"hit": [], "miss": []})
for r in resp_rows:
    k = (r["si"], r["unit"], r["channel"], r["current"])
    if k not in lab_of or r["phase"] is None:
        continue
    key = (r["week"], "PL" if lab_of[k] else "NPL")
    trial_pool[key]["hit"].extend(r["trial_delta"][r["trial_hit"]].tolist())
    trial_pool[key]["miss"].extend(r["trial_delta"][~r["trial_hit"]].tolist())
print("trials per week and label (hit, miss):", {k: (len(v["hit"]), len(v["miss"])) for k, v in sorted(trial_pool.items())})
res_kl = {}
for reading, wins, frac in [("k1", 0.01, 0.4), ("k2", 0.025, 0.8)]:
    rngk = np.random.default_rng(10)
    kl = {k: icms.kl_bootstrap(v["hit"], v["miss"], rngk, wins, frac) for k, v in sorted(trial_pool.items())}
    def pool(phase, lab):
        return np.concatenate([kl[(w, lab)] for w in (icms.EARLY_WEEKS if phase == "early" else icms.LATE_WEEKS)
                               if (w, lab) in kl])
    tests = {"PL early vs late": icms.mwu(pool("early", "PL"), pool("late", "PL")),
             "NPL early vs late": icms.mwu(pool("early", "NPL"), pool("late", "NPL")),
             "NPL vs PL early": icms.mwu(pool("early", "PL"), pool("early", "NPL")),
             "NPL vs PL late": icms.mwu(pool("late", "PL"), pool("late", "NPL"))}
    q = icms.fdr_bh([t["p"] for t in tests.values()])
    print(f"\nreading {reading}: median KL per week and label:",
          {f"w{w} {l}": round(float(np.median(v)), 4) for (w, l), v in kl.items()})
    for (name, t), qq in zip(tests.items(), q):
        print(f"  {name}: medians {t['med1']:.4f} vs {t['med2']:.4f} (n={t['n1']}, {t['n2']}), FDR p = {qq:.3g}, r = {t['r']:.2f}")
    res_kl[reading] = (tests, q)

**Result: not reproduced.** NPL responses separate hit from miss more than PL in both phases (reading k1, median KL 0.2089 against 0.0403 early and 0.2204 against 0.0279 late, FDR p = 0), and NPL separation rises over training (0.2089 to 0.2204, FDR p = 5.26e-08, r = 0.09), but PL separation falls (0.0403 to 0.0279, FDR p = 3e-192, r = -0.49). Under reading k2 both fall (PL 0.0586 to 0.0348, NPL 0.4185 to 0.2192), and NPL exceeds PL in both phases.

## fig6d-npl-hit-miss-divergence

> PL responses show no significant hit-miss divergence in firing rate at any time, while NPL
> responses diverge from 65 ms after ICMS onset early in training and from 35 ms late in
> training, both before movement onset.

Each row is one modulated response's trial-averaged rate for hit and for miss trials (responses
with at least one of each), from train onset to 700 ms. Labels are permuted by swapping a
response's hit and miss rows, which permutes the labels over rows while keeping each response's
pair (the code's permutation of hit and miss labels over rows; the exact scheme is not in the
paper). 1000 permutations. Readings (b1) 10 ms bins and (b2) 5 ms bins, onset at the bin centre;
(o1) first and (o2) last significant cluster.

In [ ]:
t0 = time.time()
res_6d = {}
for bin_s, bl in [(0.010, "b1"), (0.005, "b2")]:
    rows_h = defaultdict(list)
    rows_m = defaultdict(list)
    for r in resp_rows:
        k = (r["si"], r["unit"], r["channel"], r["current"])
        if k not in lab_of or r["phase"] is None or r["trial_hit"].sum() == 0 or (~r["trial_hit"]).sum() == 0:
            continue
        spk = sessions[r["si"]]["spikes"][r["unit"]]
        key = (r["phase"], "PL" if lab_of[k] else "NPL")
        rows_h[key].append(icms.psth(spk, r["starts"][r["trial_hit"]], bin_s))
        rows_m[key].append(icms.psth(spk, r["starts"][~r["trial_hit"]], bin_s))
    for key in sorted(rows_h):
        rngc = np.random.default_rng(20)
        H, Mm = np.array(rows_h[key]), np.array(rows_m[key])
        cl, tstat = icms.cluster_test(H, Mm, rngc, n_perm=1000)
        sig = [c for c in cl if c["p"] < 0.05]
        on1 = (sig[0]["start_bin"] + 0.5) * bin_s * 1000 if sig else np.nan
        on2 = (sig[-1]["start_bin"] + 0.5) * bin_s * 1000 if sig else np.nan
        res_6d[(bl, key)] = (len(H), sig, on1, on2)
        print(f"{bl} {key[0]} {key[1]}: rows {len(H)}, clusters {len(cl)}, significant "
              f"{[(round((c['start_bin'] + 0.5) * bin_s * 1000, 1), round((c['end_bin'] + 0.5) * bin_s * 1000, 1), round(c['mass'], 1), c['p']) for c in sig]}; "
              f"onset first {on1} ms, last {on2} ms")
print(f"cluster tests in {time.time() - t0:.0f} s")

**Result: not reproduced.** PL responses diverge between hit and miss in both phases (with 10 ms bins, significant clusters from 365 ms early and from 5 ms late; with 5 ms bins from 107.5 and 2.5 ms). With 10 ms bins the first NPL cluster starts at 75 ms early, at the edge of 65 +- 10 ms, and at 5 ms late (a single bin), outside 35 +- 10 ms; the last NPL cluster starts at 385 ms early and 45 ms late. With 5 ms bins, 87.5 and 7.5 ms (first) and 637.5 and 437.5 ms (last). No reading has both NPL onsets within tolerance, though late is earlier than early under the first-cluster readings.

## fig6f-population-coupling

> PL population coupling increases from early to late while NPL coupling stays stable, and NPL
> units have higher population coupling than PL units in both phases.

In [ ]:
def coupling_tests(rows_c, label):
    C = pd.DataFrame(rows_c)
    C = C[C.phase.notna() & C.coupling.notna()]
    C["lab"] = np.where(C.pl.astype(bool), "PL", "NPL")
    tests = {"PL early vs late": icms.mwu(C[(C.lab == "PL") & (C.phase == "early")].coupling,
                                          C[(C.lab == "PL") & (C.phase == "late")].coupling),
             "NPL early vs late": icms.mwu(C[(C.lab == "NPL") & (C.phase == "early")].coupling,
                                           C[(C.lab == "NPL") & (C.phase == "late")].coupling),
             "NPL vs PL early": icms.mwu(C[(C.lab == "PL") & (C.phase == "early")].coupling,
                                         C[(C.lab == "NPL") & (C.phase == "early")].coupling),
             "NPL vs PL late": icms.mwu(C[(C.lab == "PL") & (C.phase == "late")].coupling,
                                        C[(C.lab == "NPL") & (C.phase == "late")].coupling)}
    q = icms.fdr_bh([t["p"] for t in tests.values()])
    print(f"{label}: values {len(C)}, range {C.coupling.min():.2f} to {C.coupling.max():.2f}")
    for (name, t), qq in zip(tests.items(), q):
        print(f"  {name}: medians {t['med1']:.3f} vs {t['med2']:.3f} (n={t['n1']}, {t['n2']}), FDR p = {qq:.3g}, r = {t['r']:.2f}")
    print("  median by mouse and label:", C.groupby([C.subject.map(icms.MOUSE), "lab"]).coupling.median().round(2).to_dict())
    return tests, q

for r in resp_rows:
    r["mod"] = r["mod50"]  # population and units: the mod50 responses
t0 = time.time()
pc_main = icms.population_coupling(sessions, resp_rows, rng_seed=30)
res_6f = coupling_tests(pc_main, "700 ms, all six mice")
print(f"in {time.time() - t0:.0f} s")

**Result: reproduced.** PL coupling rises from early to late (median -0.134 to 0.481, FDR p = 0.000671, r = 0.11), NPL coupling does not change (1.593 to 1.468, FDR p = 0.303), and NPL exceeds PL early (FDR p = 6.08e-22, r = 0.42) and late (FDR p = 3.83e-22, r = 0.30). The n (PL 503 and 906, NPL 270 and 581) are close to the paper's (494 and 899, 263 and 553). NPL is above PL in every mouse.

## figs9a-coupling-700ms

> In the 700 ms window of fig. S9A, PL population coupling increases from early to late while NPL
> coupling does not change, and PL coupling is lower than NPL coupling in both phases.

Readings (a) as fig6f without Mouse 6, (b) the population built from the other modulated PL
units only.

In [ ]:
pc_a = [r for r in pc_main if r["subject"] != "ICMS83"]
res_s9a_a = coupling_tests(pc_a, "(a) without Mouse 6")
pc_b = icms.population_coupling(sessions, resp_rows, rng_seed=31, pop_label=True)
res_s9a_b = coupling_tests(pc_b, "(b) population of PL units only")

**Result: reproduced.** Without Mouse 6 (reading a), PL coupling rises (-0.206 to 0.513, FDR p = 7.76e-06), NPL does not change (FDR p = 0.366), and PL is below NPL early (FDR p = 2.48e-22) and late (FDR p = 5.6e-19). The PL n (453, 769) are near the legend's 446 and 763. Reading (b) gives the same four results (FDR p = 4.84e-05, 0.24, 1.49e-22, 2.75e-19).

## figs9-coupling-window-and-control (120 ms part)

> NPL coupling exceeds PL coupling also within the 120 ms premotion window, and the difference is
> absent in control animals without behavioural learning.

The control part is run at the end, with the control animals. Here, spikes in the first 120 ms of
each train, a 10-spike minimum, labels from the 700 ms window.

In [ ]:
pc_120 = icms.population_coupling(sessions, resp_rows, rng_seed=32, window=0.12, min_spikes=10)
res_s9_120 = coupling_tests(pc_120, "120 ms window")

**Result (120 ms part): holds.** In the first 120 ms, NPL coupling exceeds PL early (0.751 against -0.520, FDR p = 1.12e-14, r = 0.35) and late (0.420 against -0.349, FDR p = 1.04e-15, r = 0.26), as the paper reports. The outcome of the claim is given with its control part, below.

## Control animals

The control files are first described: their stimulation protocol, their imaging tables and their
units. The control claims are then tested as far as the files allow, with the paper's control code
(processing/control at d1922d1) for what the paper does not say.

In [ ]:
import re

ctrl = [a for a in assets if a["path"].split("/")[0][4:] not in icms.BEHAVIOUR]
crow = []
stim_desc = set()
trial_desc = set()
for a in ctrl:
    f = icms.open_asset(a["asset_id"])
    d = {"file": a["path"].split("/")[1], "date": f["session_start_time"][()].decode()[:10],
         "subject": f["general/subject/subject_id"][()].decode()}
    if "intervals/electrical_stimulation" in f:
        st = f["intervals/electrical_stimulation"]
        stim_desc.add(str(st.attrs["description"]))
        d["pulses_per_burst"] = ",".join(map(str, np.unique(st["pulse_count"][()])))
        d["freq_hz"] = ",".join(map(str, np.unique(st["frequency_hz"][()])))
        d["bursts_per_trial"] = int(np.max(st["burst_index_in_trial"][()]) + 1) if "burst_index_in_trial" in st else None
        d["burst_ms"] = float(np.round(np.median(st["stop_time"][()] - st["start_time"][()]) * 1000, 1))
        _s, _t = st["start_time"][()], icms.control_trial_ids({k: st[k][()] for k in ["start_time", "burst_index_in_trial"]})
        _o = np.argsort(_s)
        _s, _t = _s[_o], _t[_o]
        _same = _t[1:] == _t[:-1]
        d["burst_onset_gap_s"] = f"{np.diff(_s)[_same].min():.3f}-{np.diff(_s)[_same].max():.3f}"
        _first = np.array([_s[_t == k].min() for k in np.unique(_t)])
        d["trial_onset_gap_s"] = f"{np.diff(_first).min():.2f}-{np.diff(_first).max():.2f}" if len(_first) > 1 else ""
        d["trials_rows"] = len(f["intervals/trials/id"]) if "intervals/trials" in f else None
        d["stim_trials"] = len(np.unique(_t))
        d["trial_index_values"] = len(np.unique(st["trial_index"][()]))
        if "intervals/trials" in f:
            trial_desc.add(str(f["intervals/trials/trial_index"].attrs.get("description", "")))
            d["trial_pulses"] = ",".join(map(str, np.unique(f["intervals/trials/n_pulses"][()])))
            d["reps"] = len(np.unique(f["intervals/trials/rep_num"][()]))
        d["units"] = len(f["units/id"]) if "units" in f else 0
        d["cell_types"] = ",".join(np.unique(f["units/cell_type"][()].astype(str))) if "units" in f else ""
    o = "processing/ophys"
    if o + "/stimulus_conditions" in f:
        g = f[o + "/ImageSegmentation_ControlSurvey/PlaneSegmentation_ControlSurvey"]
        act = g["activation"][()]
        cur = f[o + "/stimulus_conditions/current_uA"][()]
        d["survey_rois"] = act.shape[0]
        d["currents_imaged"] = ",".join(str(int(c)) for c in np.unique(cur))
        d["activated_at_5uA_per_channel"] = ",".join(str(int(v)) for v in act[:, cur == 5].sum(0))
        ip = f["general/optophysiology/ImagingPlane_ControlSurvey"]
        d["grid_spacing_given"] = "grid_spacing" in ip
        _pl = re.search(r"(\d+) planes", ip["description"][()].decode())
        d["planes_in_description"] = int(_pl.group(1)) if _pl else None
        d["stim_contacts_at_origin"] = int(((f[o + "/stim_electrode_positions/x"][()] == 0) &
                                            (f[o + "/stim_electrode_positions/y"][()] == 0)).sum())
    f.close()
    crow.append(d)
CT = pd.DataFrame(crow).sort_values(["subject", "date"])
print(CT.to_string())
print("\ndescriptions of the control electrical_stimulation tables:", sorted(stim_desc))
print("descriptions of the control trials tables' trial_index:", sorted(trial_desc),
      "| trials per distinct trial_index value: trials_rows / trial_index_values above")
print("\ncontrol mice:", CT.subject.nunique(), "| control files:", len(CT), "| with survey imaging:",
      int(CT.survey_rois.notna().sum()), "| with units:", int((CT.get("units", pd.Series(dtype=float)).fillna(0) > 0).sum()))

## figs2de-control-stable (count half)

> In control animals stimulated without behavioural training, activation density within 200 um and
> the number of activated neurons do not change from early to late weeks.

The control survey files store each ROI's binary activation per stimulation channel and current;
the number of activated ROIs per channel and current is their sum. Density within 200 um needs ROI
distances from the stimulation site, which need the voxel size and the contact positions that the
files do not give (data check above), so only the count half is run. The 800 um limit of the count
cannot be applied for the same reason; all activated ROIs are counted. Weeks are counted from each
control animal's first imaging session (common method). The controls were imaged at 2, 5, 7 and
10 uA, so of the 4 to 6 uA that reading (b) pools only 5 uA exists. Readings: (a) 5 uA, late weeks
2 and 3, (b) 5 uA, late weeks 2 to 4; each on channel x session values and on channel x week
medians (the units of fig2c); all currents pooled is a further variant.

In [ ]:
crows = []
for a in ctrl:
    f = icms.open_asset(a["asset_id"])
    o = "processing/ophys"
    if o + "/stimulus_conditions" in f:
        act = f[o + "/ImageSegmentation_ControlSurvey/PlaneSegmentation_ControlSurvey/activation"][()]
        cur = f[o + "/stimulus_conditions/current_uA"][()]
        chn = f[o + "/stimulus_conditions/stim_channel"][()].astype(str).astype(int)
        sub, date = f["general/subject/subject_id"][()].decode(), f["session_start_time"][()].decode()[:10]
        for j in range(len(cur)):
            crows.append({"subject": sub, "date": date, "channel": chn[j], "current": cur[j], "count": int(act[:, j].sum())})
    f.close()
CC = pd.DataFrame(crows)
_cs = [{"subject": a, "date": b} for a, b in CC[["subject", "date"]].drop_duplicates().values]
icms.add_weeks(_cs)
CC["week"] = CC.set_index(["subject", "date"]).index.map({(s["subject"], s["date"]): s["week"] for s in _cs})
print("control imaging sessions per mouse:", CC.groupby("subject").date.nunique().to_dict(),
      "| channel-current values per week:", CC.week.value_counts().sort_index().to_dict())
print("count range:", CC["count"].min(), "to", CC["count"].max(), "| currents:", sorted(CC.current.unique()))
res_s2de = {}
for reading, late, curs in [("(a) 5 uA, late weeks 2-3", (2, 3), (5.0,)), ("(b) 5 uA, late weeks 2-4", (2, 3, 4), (5.0,)),
                            ("variant, all currents, late weeks 2-4", (2, 3, 4), (2.0, 5.0, 7.0, 10.0))]:
    d = CC[CC.current.isin(curs) & (CC.week <= 4)]
    t1 = icms.mwu(d.loc[d.week.isin(icms.EARLY_WEEKS), "count"], d.loc[d.week.isin(late), "count"])
    dm = d.groupby(["subject", "channel", "current", "week"], as_index=False)["count"].median()
    t2 = icms.mwu(dm.loc[dm.week.isin(icms.EARLY_WEEKS), "count"], dm.loc[dm.week.isin(late), "count"])
    res_s2de[reading] = (t1, t2)
    for unit, t in [("channel x session", t1), ("channel x week medians", t2)]:
        print(f"{reading}, {unit}: early median {t['med1']:.1f} (n={t['n1']}), late median {t['med2']:.1f} (n={t['n2']}), "
              f"p = {t['p']:.3g}, r = {r_from_p(t['p'], t['n1'], t['n2']):.2f}")
print("median count at 5 uA by mouse and week:")
print(CC[CC.current == 5].pivot_table(index="subject", columns="week", values="count", aggfunc="median"))

**Result: blocked (density cannot be formed); the count half does not show stability.** At 5 uA the number of activated ROIs per channel and session falls from a median of 10 early to 2 late (reading a, late weeks 2 and 3: p = 0.0146, r = 0.23; reading b, weeks 2 to 4: p = 0.00573), and with channel x week medians p = 0.0816 (a) and 0.0261 (b). So the count test gives P >= 0.05 under one of the four combinations, where the paper reports p = 0.066 (n 42 and 26; here 63 and 43 to 67). The density half, which the criterion also requires, cannot be formed.

## figs7-control-no-change

> In control animals stimulated without behavioural training, neither PL nor NPL responses change in
> modulation, time to maximum firing rate or PL peak spike probability.

The paper says only that the controls had similar stimulation parameters, but its code runs the
control units through the behavioural response classes with a 20 ms pulse window
(processing/control/postprocessing/create_session_responses.py) and tests them at 5 uA with its own
filters (python/fig_s7/generate_figure.py: P < 0.05, z > 0, baseline at least 0.5 Hz, more than 5
trains, more than 15 pulse-window spikes, weeks below 5). That code is followed (`icms.control_responses`):
each 200 ms burst is a train, the baseline is the 700 ms before it, pulse-locking uses 20 ms windows.
The code's timing file (control_timing_params.json) is not in the repository, so the 20 ms window
comes from the code's docstring and the blanking (1.5 ms and 0.5 ms) is the behavioural one; time to
maximum is taken over the 200 ms burst. These are assumptions. Weeks count from each animal's first
electrophysiology session, the first dates the code lists (python/utils/config.py, CONTROL_SESSIONS).
Readings: (a) 5 uA (code) and (b) all currents pooled; the controls have no 4 or 6 uA.

The bursts of a trial start 0.5 s apart (control data check above), so the 700 ms baseline of the
second and third burst holds the burst before it. A baseline with stimulation in it can hide a change,
and this is a claim of absence, so a variant takes every burst's baseline from the 700 ms before the
first burst of its trial (trials are 10 s apart), with the same pulse-locking draws. The claim is
judged on the code's test, as followed above (protocol, claims of absence); the variant is shown.

In [ ]:
cephys = []
for a in ctrl:
    f = icms.open_asset(a["asset_id"])
    has_units = "units" in f
    f.close()
    if has_units:
        cephys.append(icms.load_control_ephys(a))
cephys.sort(key=lambda s: (s["subject"], s["date"]))
icms.add_weeks(cephys)
print("control ephys sessions with units:", len(cephys), "| mice:", sorted({s["subject"] for s in cephys}))
print([(s["subject"], s["date"], s["week"], len(s["spikes"])) for s in cephys])
t0 = time.time()
CR = pd.DataFrame(icms.control_responses(cephys, rng_seed=40, currents=None))
CR["mod"] = ((CR.p < 0.05) & (CR.z > 0) & (CR.base_rate >= 0.5) & (CR.num_trials > 5) & (CR.num_spikes > 15)
             & (CR.week < 5))
print(f"control responses: {len(CR)} in {time.time() - t0:.0f} s; bursts per response {CR.num_trials.min()} to "
      f"{CR.num_trials.max()}; modulated {int(CR['mod'].sum())}")
res_s7 = {}
for reading, curs in [("(a) 5 uA", (5.0,)), ("(b) all currents", tuple(sorted(CR.current.unique())))]:
    d = CR[CR["mod"] & CR.current.isin(curs)]
    print(f"\nreading {reading}: modulated {len(d)}, PL {int(d.pl.sum())}; by phase and label:",
          pd.crosstab(d.phase, d.pl.map({True: "PL", False: "NPL"})).to_dict())
    out = {}
    for lab, g, col in [("PL", d[d.pl], "z"), ("NPL", d[~d.pl], "z"), ("PL", d[d.pl], "tmax_ms"),
                        ("NPL", d[~d.pl], "tmax_ms"), ("PL", d[d.pl], "peak_prob")]:
        t = icms.mwu(g.loc[g.phase == "early", col], g.loc[g.phase == "late", col])
        out[(lab, col)] = t
        print(f"  {lab} {col}: early median {t['med1']:.3f} (n={t['n1']}), late median {t['med2']:.3f} (n={t['n2']}), "
              f"p = {t['p']:.3g}, r = {t['r']:.2f}")
    res_s7[reading] = out
# How many bursts have an earlier burst inside their 700 ms baseline (bursts 0.5 s apart, cell above)
_n_in, _n_all = 0, 0
for s in cephys:
    st = s["stim"]
    keep = st["stop_time"] < s["ephys_end"]
    on, off = st["start_time"][keep], st["stop_time"][keep]
    for a in on:
        _n_all += 1
        _n_in += bool(np.any((off > a - icms.CONTROL_PRE_S) & (on < a)))
print(f"\nbursts whose 700 ms baseline holds an earlier burst: {_n_in} of {_n_all}")
# Variant: the baseline of every burst is the 700 ms before the first burst of its trial (no stimulation
# in it). Same seed, so the pulse-locking draws and PL labels are those of the main reading.
CRt = pd.DataFrame(icms.control_responses(cephys, rng_seed=40, currents=None, baseline="trial"))
CRt["mod"] = ((CRt.p < 0.05) & (CRt.z > 0) & (CRt.base_rate >= 0.5) & (CRt.num_trials > 5) & (CRt.num_spikes > 15)
              & (CRt.week < 5))
print("PL labels identical to the main reading:", bool((CRt.pl.values == CR.pl.values).all()))
dt = CRt[CRt["mod"] & (CRt.current == 5)]
print(f"variant, baseline before the trial's first burst, reading (a) 5 uA: modulated {len(dt)}; by phase and label:",
      pd.crosstab(dt.phase, dt.pl.map({True: "PL", False: "NPL"})).to_dict())
res_s7_trial = {}
for lab, g, col in [("PL", dt[dt.pl], "z"), ("NPL", dt[~dt.pl], "z"), ("PL", dt[dt.pl], "tmax_ms"),
                    ("NPL", dt[~dt.pl], "tmax_ms"), ("PL", dt[dt.pl], "peak_prob")]:
    t = icms.mwu(g.loc[g.phase == "early", col], g.loc[g.phase == "late", col])
    res_s7_trial[(lab, col)] = t
    print(f"  {lab} {col}: early median {t['med1']:.3f} (n={t['n1']}), late median {t['med2']:.3f} (n={t['n2']}), "
          f"p = {t['p']:.3g}, r = {t['r']:.2f}")
d = CR[CR["mod"] & (CR.current == 5)]
print("\n5 uA modulated responses by mouse and phase:", pd.crosstab(d.subject, d.phase).to_dict())
print("z range", d.z.min().round(2), "to", d.z.max().round(2), "| time to max range", d.tmax_ms.min(), "to", d.tmax_ms.max(), "ms")

**Result: reproduced, resting on the assumptions about the control code's timing.** At 5 uA (reading a, the code's current), 111 control responses are modulated, PL 32 early and 19 late, NPL 27 and 33 (paper 29 and 12, 30 and 29). None of the five tests is significant: PL modulation p = 0.946, NPL modulation p = 0.122, PL time to maximum p = 0.112, NPL time to maximum p = 0.399, PL peak spike probability p = 0.182. Pooling all currents (reading b, n 59/73 PL and 74/103 NPL, further from the paper's) gives NPL modulation p = 0.00809 and PL time to maximum p = 0.0352. The criterion judges under the reading whose n is closest to the paper's, (a). The 700 ms baseline holds an earlier burst for 5863 of 8795 bursts. With the baseline taken before each trial's first burst (variant), NPL modulation rises (0.334 to 0.988, p = 0.0059, r = 0.43; n 29 and 28) and the other four tests stay above 0.05 (PL modulation p = 0.744, PL time to maximum p = 0.957, NPL time to maximum p = 0.44, PL peak spike probability p = 0.07). So the absence of a change in NPL modulation holds with the code's baseline and not with a baseline free of stimulation. PL time to maximum also moves the other way from the paper's: here it rises (5 to 55 ms, r = 0.25), where the paper reports r = -0.39.

## figs9-coupling-window-and-control (control part)

Coupling in the control animals at 5 uA (fig. S9C legend), early against late, PL against NPL, FDR
over the two tests. The paper's control code (processing/pop_coupling/pop_coupling_control.py)
concatenates, per channel and current, segments from the first to the last pulse of each row of its
trial table, builds the population from every unit with a response at that condition (not only the
modulated ones) and labels each unit PL or NPL by its call at that condition; units need 50 spikes.
Its trial table is not public, so whether a row is a trial (3 bursts) or a burst is open: readings
(s1) trial segments and (s2) burst segments. The stored trial_index restarts in each recording block
(the trials table's rep_num), so trials are numbered from the bursts' burst_index_in_trial in time
order (`icms.control_trial_ids`); the previous revision grouped by trial_index and so joined trials of different
blocks into one segment. Populations: (p1) all units (code), (p2) the modulated
units only (as fig6f).

In [ ]:
res_s9c = {}
for seg in ["s1 trial", "s2 burst"]:
    for popr in ["p1 all units", "p2 modulated"]:
        rngc9 = np.random.default_rng(41)
        out = []
        for (si, ch, cur), g in CR[(CR.current == 5) & (CR.week < 5)].groupby(["si", "channel", "current"]):
            s = cephys[si]
            st = s["stim"]
            m = (st["stim_channel"] == ch) & (st["current_uA"] == cur) & (st["stop_time"] < s["ephys_end"])
            tid = icms.control_trial_ids(st)  # trial_index restarts in each block (cell above)
            if seg.startswith("s2"):
                segs = [(a, a + 0.18) for a in np.sort(st["start_time"][m])]
            else:
                segs = [(st["start_time"][m & (tid == k)].min(),
                         st["start_time"][m & (tid == k)].max() + 0.18) for k in np.unique(tid[m])]
            use = g if popr.startswith("p1") else g[g["mod"]]
            if len(segs) < 3 or len(use) < 2:
                continue
            pc = icms.coupling_on_segments({u: s["spikes"][u] for u in use.unit}, sorted(segs), rngc9)
            lab = dict(zip(use.unit, use.pl))
            out += [{"phase": g.phase.iloc[0], "subject": s["subject"], "pl": lab[u], "c": v} for u, v in pc.items()]
        O = pd.DataFrame(out)
        O = O[np.isfinite(O.c)]
        tests = [icms.mwu(O[O.pl & (O.phase == ph)].c, O[~O.pl & (O.phase == ph)].c) for ph in ["early", "late"]]
        q = icms.fdr_bh([t["p"] for t in tests])
        res_s9c[(seg, popr)] = (tests, q)
        print(f"{seg}, {popr}: " + "; ".join(
            f"{ph} PL {t['med1']:.2f} (n={t['n1']}) vs NPL {t['med2']:.2f} (n={t['n2']}), FDR p = {qq:.3g}, r = {t['r']:.2f}"
            for ph, t, qq in zip(["early", "late"], tests, q)))

**Result: reproduced, resting on the assumption about the control trial segments.** The 120 ms part holds (above). In the controls at 5 uA, with the code's population of all units and segments from the first to the last pulse of each trial (s1, p1), PL and NPL coupling do not differ early (2.03 against 1.56, n 45 and 48, FDR p = 0.431) or late (2.46 against 1.58, n 31 and 90, FDR p = 0.134); the paper reports n 49 and 64, 25 and 128, FDR p = 0.48. With the modulated units as population (s1, p2), FDR p = 0.99 in both phases. With burst segments (s2), the early difference is significant with all units (FDR p = 0.00428, n 41 and 18) and not with modulated units (0.173). The trial reading is taken because the criterion says the full train, the dataset's own trials table has one row per trial of 3 bursts (as many rows as such trials in 14 of the 15 files with stimulation, and n_pulses 30, the pulses of 3 bursts, in most; control data check), and its n are closer to the paper's in both phases (45/48 and 31/90 against 41/18 and 28/46 for bursts).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2a-threshold-decrease | reproduced |
| fig2a-steepest-decline-early | reproduced (rests on the criterion's reading of the figure) |
| figs1e-response-times-decrease | reproduced |
| figs5c-wheel-onset-decrease | reproduced |
| fig4-unit-counts | not reproduced |
| figs3b-modulation-increase-all | reproduced |
| figs3c-latency-all | partially reproduced |
| figs3d-modulation-stable-at-threshold | reproduced |
| fig4g-modulated-count-mouse1 | reproduced |
| fig4ij-pyramidal-modulation-increase | reproduced |
| fig4ij-interneurons-little-change | reproduced |
| fig5d-pulse-locking-stable-across-currents | not reproduced |
| fig5e-pyramidal-fraction-pl | partially reproduced |
| figs4b-pl-latency-distance | partially reproduced |
| fig5fg-pl-modulation-increase | reproduced |
| figs4de-npl-no-change | reproduced |
| fig5h-pl-spike-probability-increase | partially reproduced |
| figs5-restricted-window-consistent | reproduced |
| fig6c-kl-divergence | not reproduced |
| fig6d-npl-hit-miss-divergence | not reproduced |
| fig6f-population-coupling | reproduced |
| figs9a-coupling-700ms | reproduced |
| figs9-coupling-window-and-control | reproduced (rests on the control trial segments) |
| figs7-control-no-change | reproduced (rests on the control code's timing) |
| fig2c, fig2b, figs2ab, fig2e (two), figs2de-behaviour, fig2g | blocked (metadata insufficient) |
| figs2de-control-stable | blocked (metadata insufficient; the count half is reported) |
| fig3b, fig3df, fig3i, fig3j (two), fig3lm, fig3n (two), figs6 | blocked (metadata insufficient) |
| fig4d-waveform-separation, figs8-tracked-units | blocked (data missing) |

How the analysis differs from the paper's. Thresholds are refitted with the Python port of
psignifit; the dataset stores none. The movement-onset threshold is taken from hit trials, as miss
trials have no response time. Pulse-locking follows the paper's code, with pulses placed 10 ms
apart from the train onsets, as no pulse times are stored; the call is random, and the claims
that rest on it near a threshold (fig5d, fig5e, fig5h) are shown under three other seeds. The
hit-miss PSTH bin width and smoothing are not in the paper or its code; two widths were run. The
stimulation contact is placed as the paper's code places it, not by the electrode table. The
control analyses follow the paper's control code, whose timing file is not public; its 700 ms
baseline holds the previous burst for most bursts, and a baseline free of stimulation is shown as a
variant. Control trials are numbered from `burst_index_in_trial`, as the stored `trial_index`
restarts in each recording block. Cell types are
the stored labels, which the files describe as the trough-to-peak classification; the rule applied
to the stored mean waveform is shown as a variant.

What was found about the dataset. The 46 sessions of weeks 0 to 4 match the paper's count, and the
ephys response counts are close to the paper's, but the unit total (795 in weeks 0 to 4) is below
the paper's 973. Behavioural files give no stimulation-site position in the imaging volume and no
activation calls, and the planar ROIs are segmented per session with no identity across sessions,
so no behavioural imaging claim can be tested. The stored `stim_channel` does not index the
electrode table's `channel_name`, though both are described as Ripple channel numbers. The control
files use 3 bursts of 10 pulses at 50 Hz, where the paper says the controls had similar stimulation
parameters. In the control files `trial_index`, described as the trial index within the session, restarts in
each recording block. Most miss trials carry no response time, Mouse 6 has no catch trials, and cell types
are spelled differently in Mouse 6.